In [ ]:
import os, sys, glob, time, json, math, warnings, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import train_test_split
import xgboost as xgb

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ----------------------------- CONFIG ---------------------------------
SMOKE_TEST    = os.environ.get("MF_SMOKE", "0") == "1"   # offline code test on synthetic stand-in data (results meaningless). Leave False on Kaggle.
FAST          = False         # True: first Kaggle run. 5 LOFO folds, 1 P1 seed, smaller models, fewer attack rows (~10-15 min on CPU)
SEED0         = 42            # base seed (also fixes the benign train/val/test split shared by all LOFO folds)
PROTOCOLS     = ["P3_LOFO", "P1_random"]   # P3 is the primary protocol
N_SEEDS_P1    = 3             # repeats of the random-split check
LOFO_N_VAL_FAMILIES = 2       # unseen families used only for threshold calibration / early stopping in each fold
LOFO_MAX_FOLDS = None         # None = all families (15 on CIC-MalMem-2022)
ALPHAS        = [0.1, 0.25, 0.5, 0.75, 1.0]   # evasion strengths (0 = clean)
PRIMARY_ALPHA = 0.5           # headline evasion strength
FPR_TARGET    = 0.01          # constrained-FPR operating point
AUG_FRAC      = 0.5           # fraction of training malware copied per technique for evasion-aware training
JITTER        = 0.05          # adversary noise (fraction of benign std)
FEASIBLE      = True          # apply the feasibility projection to every perturbed row
FEAS_HOLD     = 0.995         # a hard constraint is adopted only if >= this fraction of TRAINING rows satisfy it
XGB_N_EST     = 300
RF_N_EST      = 200
XGB_DEVICE    = "cpu"         # tabular data is small; "cuda" gives no real gain here
# ---- adaptive attack ----
ADAPT         = True
ADAPT_N       = 200           # malware rows attacked per fold
ADAPT_KMAX    = 24            # max number of features the adversary may change
ADAPT_KS      = [1, 2, 4, 8, 12, 16, 24]
SENS_JITTER   = [0.0, 0.05, 0.2, 0.5]
# ---- Model B (LLM) ----
RUN_LLM       = True
LLM_BACKEND   = "hf"          # "hf" (real model; needs GPU) or "mock" (plumbing test only)
LLM_MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"   # fallback if VRAM/time is a problem: "Qwen/Qwen2.5-3B-Instruct"
LLM_4BIT      = True
LLM_N_PER_FAMILY = 20         # malware rows per family used for LLM test (and again, disjoint, for LLM calibration)
LLM_N_BENIGN  = 150           # benign rows for LLM test (and again for calibration)
LLM_BATCH     = 8
LLM_CONDITIONS = [("Injection", 0.5), ("Cross-view", 0.5), ("Combined", 0.5), ("Combined", 1.0)]
# ----------------------------------------------------------------------
if FAST:
    LOFO_MAX_FOLDS, N_SEEDS_P1, XGB_N_EST, RF_N_EST, ADAPT_N = 5, 1, 200, 100, 100
if SMOKE_TEST:
    LOFO_MAX_FOLDS, N_SEEDS_P1, XGB_N_EST, RF_N_EST, ADAPT_N = 3, 1, 60, 40, 40
    ADAPT_KMAX, ADAPT_KS, LLM_BACKEND, LLM_N_PER_FAMILY, LLM_N_BENIGN = 8, [1, 2, 4, 8], "mock", 10, 60

ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path("/kaggle/working/outputs") if ON_KAGGLE else Path("outputs")
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Kaggle:", ON_KAGGLE, "| output dir:", OUT.resolve(), "| FAST:", FAST, "| SMOKE:", SMOKE_TEST)

# Display names. Colour = documented categorical slots 1-6 in order (blue, orange, aqua, yellow, magenta, green);
# every model ALSO has its own marker, line style and hatch, so figures survive greyscale printing and colour-vision deficiency.
M_AT, M_A, M_RF, M_LR, M_RULE, M_B = ("Model A-AT (evasion-aware)", "Model A (XGBoost)", "Random forest",
                                      "Logistic regression", "Rule baseline", "Model B (LLM)")
MODELS = [M_AT, M_A, M_RF, M_LR, M_RULE]          # legend order = colour-slot order
STYLE = {M_AT:   dict(color="#2a78d6", marker="o", ls="-",  hatch=""),
         M_A:    dict(color="#eb6834", marker="s", ls="--", hatch="//"),
         M_RF:   dict(color="#1baf7a", marker="^", ls="-.", hatch="xx"),
         M_LR:   dict(color="#eda100", marker="v", ls=":",  hatch=".."),
         M_RULE: dict(color="#e87ba4", marker="D", ls=(0, (5, 1, 1, 1)), hatch="++"),
         M_B:    dict(color="#008300", marker="P", ls=(0, (1, 1)), hatch="oo")}
TAU_KINDS = ["Youden", "FPR<=1%"]

mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.2,
    "lines.markersize": 4, "savefig.dpi": 300, "figure.dpi": 110, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.4,
    "pdf.fonttype": 42, "ps.fonttype": 42})   # fonttype 42 = embedded TrueType (IEEE PDF eXpress requirement)

## 1. Data loading
Attach the Kaggle dataset **`jlcole/cic-malmem-2022`** (file `Obfuscated-MalMem2022.csv`) via *Add Input*. The loader searches `/kaggle/input` recursively, so the exact folder name does not matter.

In [ ]:
FEATURE_NAMES_55 = (
    [f"pslist.{c}" for c in ["nproc","nppid","avg_threads","nprocs64bit","avg_handlers"]] +
    [f"dlllist.{c}" for c in ["ndlls","avg_dlls_per_proc"]] +
    [f"handles.{c}" for c in ["nhandles","avg_handles_per_proc","nport","nfile","nevent","ndesktop","nkey","nthread",
                              "ndirectory","nsemaphore","ntimer","nsection","nmutant"]] +
    [f"ldrmodules.{c}" for c in ["not_in_load","not_in_init","not_in_mem","not_in_load_avg","not_in_init_avg","not_in_mem_avg"]] +
    [f"malfind.{c}" for c in ["ninjections","commitCharge","protection","uniqueInjections"]] +
    [f"psxview.not_in_{v}" for v in ["pslist","eprocess_pool","ethread_pool","pspcid_list","csrss_handles","session","deskthrd"]] +
    [f"psxview.not_in_{v}_false_avg" for v in ["pslist","eprocess_pool","ethread_pool","pspcid_list","csrss_handles","session","deskthrd"]] +
    ["modules.nmodules"] +
    [f"svcscan.{c}" for c in ["nservices","kernel_drivers","fs_drivers","process_services","shared_process_services",
                              "interactive_process_services","nactive"]] +
    [f"callbacks.{c}" for c in ["ncallbacks","nanonymous","ngeneric"]])
assert len(FEATURE_NAMES_55) == 55

FAMILIES = {"Ransomware": ["Conti","MAZE","Pysa","Ako","Shade"],
            "Spyware":    ["180Solutions","CoolWebSearch","Gator","Transponder","TIBS"],
            "Trojan":     ["Zeus","Emotet","Refroso","Scar","Reconyc"]}
HANDLE_TYPES = ["nport","nfile","nevent","ndesktop","nkey","nthread","ndirectory","nsemaphore","ntimer","nsection","nmutant"]
PSX_VIEWS = ["pslist","eprocess_pool","ethread_pool","pspcid_list","csrss_handles","session","deskthrd"]

def make_synthetic_standin(n_benign=6000, n_per_family=300, seed=0):
    """Schema-compatible synthetic data with the SAME internal structure as the real feature set (counts <= totals, ratios =
    count / denominator, averages = total / nproc). Used ONLY to test the code path offline. Never report results from it."""
    rng = np.random.default_rng(seed)
    def gen(n, e):
        d = {}
        nproc = rng.poisson(60, n) + 20
        d["pslist.nproc"] = nproc; d["pslist.nppid"] = np.minimum(nproc, rng.poisson(0.3 * nproc))
        d["pslist.avg_threads"] = rng.lognormal(1.6, 0.25, n); d["pslist.nprocs64bit"] = np.zeros(n)
        d["pslist.avg_handlers"] = rng.lognormal(4.5, 0.5, n) * (1 + 0.5 * e[3])
        avg_dll = rng.lognormal(3.4, 0.2, n); ndlls = np.round(avg_dll * nproc)
        d["dlllist.ndlls"] = ndlls; d["dlllist.avg_dlls_per_proc"] = ndlls / nproc
        typed = {t: rng.poisson(nproc * s * (1 + e[3] * u)) for t, s, u in
                 zip(HANDLE_TYPES, [0, 18, 8, 0.5, 5, 4, 1, 3, 1, 6, 2], [0, .8, .2, 0, .1, .1, 0, .2, 0, .5, 0])}
        other = rng.poisson(nproc * 25)
        nh = sum(typed.values()) + other
        for t, v in typed.items(): d[f"handles.{t}"] = v
        d["handles.nhandles"] = nh; d["handles.avg_handles_per_proc"] = nh / nproc
        den_l = ndlls + rng.poisson(30, n)
        for v, b in zip(["load", "init", "mem"], [6, 8, 6]):
            c = np.minimum(den_l, rng.poisson(b + 6 * e[2], n)); d[f"ldrmodules.not_in_{v}"] = c; d[f"ldrmodules.not_in_{v}_avg"] = c / den_l
        ninj = rng.poisson(0.3 + 3.0 * e[0], n)
        d["malfind.ninjections"] = ninj; d["malfind.uniqueInjections"] = rng.binomial(ninj, 0.7)
        d["malfind.commitCharge"] = ninj * rng.integers(1, 20, n); d["malfind.protection"] = ninj * rng.integers(1, 3, n)
        for v in PSX_VIEWS:
            c = np.minimum(nproc, rng.poisson(0.4 + 1.5 * e[1], n)); d[f"psxview.not_in_{v}"] = c; d[f"psxview.not_in_{v}_false_avg"] = c / nproc
        d["modules.nmodules"] = rng.poisson(150 + 20 * e[4], n)
        ns = rng.poisson(280 + 30 * e[4], n); kd = rng.binomial(ns, 0.45); fs = rng.binomial(ns - kd, 0.05); ps = rng.binomial(ns, 0.3)
        d["svcscan.nservices"] = ns; d["svcscan.kernel_drivers"] = kd; d["svcscan.fs_drivers"] = fs; d["svcscan.process_services"] = ps
        d["svcscan.shared_process_services"] = rng.binomial(ps, 0.5); d["svcscan.interactive_process_services"] = rng.binomial(ps, 0.1)
        d["svcscan.nactive"] = rng.binomial(ns, 0.5)
        nc = rng.poisson(60 + 8 * e[4], n); d["callbacks.ncallbacks"] = nc
        d["callbacks.nanonymous"] = rng.binomial(nc, 0.05 + 0.1 * e[4]); d["callbacks.ngeneric"] = rng.binomial(nc, 0.3)
        return pd.DataFrame({c: np.asarray(d[c], float) for c in FEATURE_NAMES_55})
    frames = [gen(n_benign, np.zeros(5))]; cat = ["Benign"] * n_benign; cls = ["Benign"] * n_benign
    for typ, fams in FAMILIES.items():
        for f in fams:
            frames.append(gen(n_per_family, rng.uniform(0.0, 1.0, 5))); cat += [f"{typ}-{f}-{i}.raw" for i in range(n_per_family)]; cls += ["Malware"] * n_per_family
    df = pd.concat(frames, ignore_index=True); df.insert(0, "Category", cat); df.insert(1, "Class", cls)
    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

def load_dataset():
    if SMOKE_TEST:
        print("!!! SMOKE_TEST: synthetic stand-in data. Results are NOT scientific results. !!!")
        return make_synthetic_standin()
    cands = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if "malmem" in p.lower()]
    cands += [p for p in glob.glob("./**/Obfuscated-MalMem2022.csv", recursive=True)]
    if not cands:
        raise FileNotFoundError("CIC-MalMem-2022 CSV not found. Kaggle: Add Input -> Datasets -> search 'cic-malmem-2022' (jlcole). "
                                "Local: place Obfuscated-MalMem2022.csv next to the notebook.")
    print("Loading:", cands[0]); return pd.read_csv(cands[0])

raw = load_dataset()
print(raw.shape); display(raw.iloc[:3, :8]) if "display" in globals() else print(raw.iloc[:3, :8])

## 2. Cleaning, labels, family groups, schema check
* Label: `Class` (`Benign` -> 0, otherwise -> 1).
* Family (the **group** for leave-one-family-out) is parsed from `Category` (e.g. `Ransomware-Conti-<hash>` -> `conti`).
* Exact duplicate rows and zero-variance columns are removed. Everything is reported so the paper's data table can be filled in directly.

In [ ]:
def parse_family(cat):
    c = str(cat).strip()
    if c.lower().startswith("benign"): return "benign"
    parts = c.split("-")
    return parts[1].strip().lower() if len(parts) > 1 else parts[0].strip().lower()

def prepare(df):
    assert {"Class", "Category"} <= set(df.columns), f"Unexpected schema, columns start with: {list(df.columns)[:8]}"
    df = df.reset_index(drop=True)
    y = df["Class"].astype(str).str.strip().str.lower().ne("benign").astype(int).values
    fam = df["Category"].map(parse_family).values
    feat = df.drop(columns=["Class", "Category"]).select_dtypes(include=[np.number]).reset_index(drop=True)
    n0 = len(feat)
    keep = ~pd.concat([feat, pd.Series(y, name="_y")], axis=1).duplicated().values
    feat, y, fam = feat[keep].reset_index(drop=True), y[keep], fam[keep]
    zero_var = [c for c in feat.columns if feat[c].nunique() <= 1]
    feat = feat.drop(columns=zero_var)
    info = dict(rows_raw=n0, duplicates_removed=int(n0 - keep.sum()), rows=len(feat), benign=int((y == 0).sum()),
                malicious=int((y == 1).sum()), n_features=feat.shape[1], zero_variance_dropped=zero_var,
                families=sorted(set(fam[y == 1])))
    return dict(X=feat.values.astype(np.float64), y=y, fam=fam, cols=list(feat.columns), info=info)

D = prepare(raw)
for k, v in D["info"].items(): print(f"{k:24s}: {v}")
if len(D["info"]["families"]) < 6:
    warnings.warn("Fewer than 6 malware families parsed from 'Category' - check parse_family(); leave-one-family-out needs >= 4 families.")
D["info"]["family_counts"] = pd.Series(D["fam"][D["y"] == 1]).value_counts().to_dict()
print(D["info"]["family_counts"])

## 3. Threat model: evasion techniques and feasibility projection
**Adversary:** knows the benign feature distribution (median) and can move the artifacts of a chosen Volatility plugin group toward it. For a malicious row *x*, targeted features *j* become
`x_j' = max(0, x_j + a * (median_benign_j - x_j) + a * N(0, (JITTER * sd_benign_j)^2))`, rounded for integer-valued features, where `a` in (0, 1] is the **evasion strength** (0 = clean, 1 = full mimicry of the benign median on those features). This is a classical *mimicry attack* abstraction, **not** a reproduction of any specific malware; the ATT&CK mapping is illustrative only.

| Technique | Features moved | Illustrative real-world analogue |
|---|---|---|
| Injection | `malfind.*` | process injection / hollowing (T1055) |
| Module-view | `ldrmodules.*` | DLL injection / module-list forging (T1055.001) |
| Cross-view | `psxview.*` | DKOM-style rootkit consistency forging (T1014) |
| Footprint | `pslist.*`, `dlllist.*`, `handles.*` | living-off-the-land blending (T1218) |
| Svc/Callback | `svcscan.*`, `callbacks.*`, `modules.*` | service / kernel-object masking (T1543, T1014) |
| Combined | all of the above | multi-technique adversary |

**Feasibility projection (new).** Independent per-feature perturbation can create rows no real memory image could produce (e.g. more unique injections than injections, or a ratio that no longer equals count / denominator). The `Projector` therefore *verifies* candidate structural constraints on the **training rows only** and adopts a constraint only if at least `FEAS_HOLD` of those rows satisfy it (ratio and per-process relations use a looser tolerance). Adopted constraints are enforced after every perturbation; the report is saved to `run_record.json`. If the report shows few adopted constraints, the real column semantics differ from the assumed ones and the projection is correspondingly weaker. Say so in the paper.

In [ ]:
SINGLE = ["Injection", "Module-view", "Cross-view", "Footprint", "Svc/Callback"]
TECHS = SINGLE + ["Combined"]
TECH_PREFIX = {"Injection": ("malfind.",), "Module-view": ("ldrmodules.",), "Cross-view": ("psxview.",),
               "Footprint": ("pslist.", "dlllist.", "handles."), "Svc/Callback": ("svcscan.", "callbacks.", "modules.")}
TECH_PREFIX["Combined"] = tuple(p for t in SINGLE for p in TECH_PREFIX[t])
COLS = D["cols"]
TECH_IDX = {t: np.array([i for i, c in enumerate(COLS) if c.startswith(TECH_PREFIX[t])], dtype=int) for t in TECHS}
for t in TECHS: print(f"{t:13s}: {len(TECH_IDX[t]):2d} features")
assert all(len(TECH_IDX[t]) > 0 for t in SINGLE), "A technique matched no columns - check the column names / prefixes printed above."

class Projector:
    """Data-driven feasibility projection. Constraints are VERIFIED on training rows before being adopted."""
    LEQ = ([("malfind.uniqueInjections", "malfind.ninjections"), ("pslist.nppid", "pslist.nproc"),
            ("svcscan.nactive", "svcscan.nservices"), ("svcscan.kernel_drivers", "svcscan.nservices"),
            ("svcscan.fs_drivers", "svcscan.nservices"), ("svcscan.process_services", "svcscan.nservices"),
            ("svcscan.shared_process_services", "svcscan.process_services"),
            ("svcscan.interactive_process_services", "svcscan.process_services"),
            ("callbacks.nanonymous", "callbacks.ncallbacks"), ("callbacks.ngeneric", "callbacks.ncallbacks")] +
           [(f"handles.{t}", "handles.nhandles") for t in HANDLE_TYPES])
    GROUPS = {"ldrmodules": [(f"ldrmodules.not_in_{v}", f"ldrmodules.not_in_{v}_avg") for v in ("load", "init", "mem")],
              "psxview": [(f"psxview.not_in_{v}", f"psxview.not_in_{v}_false_avg") for v in PSX_VIEWS]}
    PERPROC = [("dlllist.ndlls", "dlllist.avg_dlls_per_proc", "pslist.nproc"), ("handles.nhandles", "handles.avg_handles_per_proc", "pslist.nproc")]

    def _den(self, X, g):
        cs, rs = [c for c, _ in g["pairs"]], [r for _, r in g["pairs"]]
        C, Rr = X[:, cs], X[:, rs]
        den = np.where((Rr > 1e-9) & (C > 0), C / np.maximum(Rr, 1e-9), np.nan)
        out = np.full(len(X), np.nan); ok = (~np.isnan(den)).any(1)
        if ok.any(): out[ok] = np.nanmedian(den[ok], axis=1)
        return np.where(np.isnan(out), g["fallback"], out)

    def fit(self, X, cols, hold=FEAS_HOLD, loose=0.9):
        ix = {c: i for i, c in enumerate(cols)}; rep = {}
        self.leq, self.hsum, self.perproc, self.groups = [], None, [], []
        for a, b in self.LEQ:
            if a in ix and b in ix:
                r = float(np.mean(X[:, ix[a]] <= X[:, ix[b]] + 1e-9)); rep[f"{a} <= {b}"] = r
                if r >= hold: self.leq.append((ix[a], ix[b]))
        parts = [ix[f"handles.{t}"] for t in HANDLE_TYPES if f"handles.{t}" in ix]
        if "handles.nhandles" in ix and len(parts) >= 2:
            r = float(np.mean(X[:, parts].sum(1) <= X[:, ix["handles.nhandles"]] + 1e-9)); rep["sum(typed handles) <= nhandles"] = r
            if r >= hold: self.hsum = (ix["handles.nhandles"], np.array(parts))
        for tot, avg, npr in self.PERPROC:
            if tot in ix and avg in ix and npr in ix:
                exp = X[:, ix[tot]] / np.maximum(X[:, ix[npr]], 1); r = float(np.mean(np.abs(X[:, ix[avg]] - exp) <= 0.05 * np.maximum(exp, 1e-6) + 1e-6))
                rep[f"{avg} = {tot} / {npr}"] = r
                if r >= 0.95: self.perproc.append((ix[tot], ix[avg], ix[npr]))
        for g, pairs in self.GROUPS.items():
            pr = [(ix[c], ix[r]) for c, r in pairs if c in ix and r in ix]
            if len(pr) < 2: continue
            gg = dict(pairs=pr, fallback=1.0); den = np.where((X[:, [r for _, r in pr]] > 1e-9) & (X[:, [c for c, _ in pr]] > 0),
                                                                X[:, [c for c, _ in pr]] / np.maximum(X[:, [r for _, r in pr]], 1e-9), np.nan)
            ok = (~np.isnan(den)).sum(1) >= 2
            if ok.sum() < 50: rep[f"{g}: ratio = count / shared denominator"] = float("nan"); continue
            dm = np.nanmean(den[ok], 1); spread = (np.nanmax(den[ok], 1) - np.nanmin(den[ok], 1)) / np.maximum(dm, 1e-9)
            r = float(np.mean(spread < 0.1)); rep[f"{g}: ratio = count / shared denominator"] = r
            if r >= loose: gg["fallback"] = float(np.nanmedian(dm)); self.groups.append(gg)
        self.report = rep; self.cols = cols; return self

    def project(self, Xn, Xo, tm):
        """Repair rows Xn (derived from original rows Xo) so adopted constraints hold. tm: bool (d,) or (n, d) = features the adversary changed."""
        X = Xn.copy(); tm = np.broadcast_to(np.asarray(tm, bool), X.shape)
        for a, b in self.leq:
            m1 = (X[:, a] > X[:, b]) & tm[:, a]; X[m1, a] = X[m1, b]
            m2 = (X[:, a] > X[:, b]) & tm[:, b] & ~tm[:, a]; X[m2, b] = X[m2, a]
        if self.hsum is not None:
            tot, parts = self.hsum; Sp = X[:, parts]; tp = tm[:, parts]
            T = (Sp * tp).sum(1); U = Sp.sum(1) - T; room = np.maximum(X[:, tot] - U, 0)
            f = np.where((Sp.sum(1) > X[:, tot]) & (T > 0), room / np.maximum(T, 1e-9), 1.0)
            X[:, parts] = np.where(tp, np.floor(Sp * f[:, None]), Sp)
        for tot, avg, npr in self.perproc:
            m = tm[:, tot] | tm[:, npr] | tm[:, avg]
            X[m, avg] = X[m, tot] / np.maximum(X[m, npr], 1)
        for g in self.groups:
            den = self._den(Xo, g)
            for c, r in g["pairs"]:
                m = tm[:, c] | tm[:, r]; X[m, r] = X[m, c] / den[m]
        return X

    def violation(self, X):
        """Boolean per row: True if any ADOPTED structural constraint is broken."""
        v = np.zeros(len(X), bool)
        for a, b in self.leq: v |= X[:, a] > X[:, b] + 1e-9
        if self.hsum is not None: v |= X[:, self.hsum[1]].sum(1) > X[:, self.hsum[0]] + 1e-9
        for tot, avg, npr in self.perproc:
            exp = X[:, tot] / np.maximum(X[:, npr], 1); v |= np.abs(X[:, avg] - exp) > 0.1 * np.maximum(exp, 1e-6) + 1e-6
        for g in self.groups:
            C, Rr = X[:, [c for c, _ in g["pairs"]]], X[:, [r for _, r in g["pairs"]]]
            den = np.where((Rr > 1e-9) & (C > 0), C / np.maximum(Rr, 1e-9), np.nan); ok = (~np.isnan(den)).sum(1) >= 2
            sp = np.zeros(len(X)); sp[ok] = (np.nanmax(den[ok], 1) - np.nanmin(den[ok], 1)) / np.maximum(np.nanmean(den[ok], 1), 1e-9)
            v |= sp > 0.2
        return v

    def adopted(self):
        n = lambda i: self.cols[i]
        return dict(leq=[f"{n(a)} <= {n(b)}" for a, b in self.leq],
                    handle_sum=None if self.hsum is None else f"sum(typed handles) <= {n(self.hsum[0])}",
                    per_process=[f"{n(a)} = {n(t)} / {n(p)}" for t, a, p in self.perproc],
                    ratio_groups=[[f"{n(r)} = {n(c)} / den" for c, r in g["pairs"]] for g in self.groups])

def fit_reference(Xb, Xall=None, cols=None):
    """Benign reference statistics (+ feasibility projector) fitted on TRAINING rows only."""
    q05, q95 = np.quantile(Xb, [0.05, 0.95], axis=0)
    is_int = np.all(np.isclose(Xb, np.round(Xb)), axis=0); med = np.median(Xb, 0)
    ref = dict(med=med, target=np.where(is_int, np.round(med), med), sd=Xb.std(0) + 1e-9, q05=q05, q95=q95, is_int=is_int, proj=None)
    if FEASIBLE and Xall is not None: ref["proj"] = Projector().fit(Xall, cols)
    return ref

def evade(Xm, idx, ref, alpha, rng, jitter=None, project=True):
    """Mimicry perturbation of malicious rows. alpha: scalar or per-row array. Benign rows are never modified."""
    jitter = JITTER if jitter is None else jitter
    a = np.asarray(alpha, dtype=float); a = a.reshape(-1, 1) if a.ndim else float(a)
    cur = Xm[:, idx]
    noise = rng.normal(0, 1, cur.shape) * ref["sd"][idx] * jitter
    new = np.maximum(cur + a * (ref["target"][idx] - cur) + a * noise, 0.0)
    ii = ref["is_int"][idx]; new[:, ii] = np.round(new[:, ii])
    Xe = Xm.copy(); Xe[:, idx] = new
    if project and ref.get("proj") is not None:
        tm = np.zeros(Xm.shape[1], bool); tm[idx] = True
        Xe = ref["proj"].project(Xe, Xm, tm)
    return Xe

## 4. Protocols, models, threshold calibration, metrics
**P3 - leave-one-family-out (LOFO), primary.** The benign rows are split once (70/10/20 train/val/test, seed-fixed, shared by all folds). In fold *f*, **all malware of family *f* is the test malware**, `LOFO_N_VAL_FAMILIES` other families form the *validation* malware (threshold calibration + early stopping only), and the remaining families train the models. Every family is held out exactly once, so there is no arbitrary 3-family test set and no family-selection luck. Fold-level metrics are the unit of analysis (n = number of folds).
**P1 - stratified random 70/10/20 (check only).** Reproduces the near-ceiling accuracy of prior work. Rows of the same family and the SMOTE-generated benign rows can leak across this split, so P1 is a ceiling / leakage reference and not evidence of generalisation.
**Thresholds** are chosen on clean validation data of *unseen* families and then **frozen**. The same tau is used for every evaded condition (otherwise the degradation is hidden by re-tuning). Two operating points: **Youden** (max TPR - FPR, centred in the score gap) and **FPR <= 1 %** (SOC-style constraint).
**Detectors.** Rule baseline (feature-space analogue of SPECTRE-style heuristics), logistic regression (log1p + standardisation), random forest, **Model A** (XGBoost), **Model A-AT** (XGBoost with evasion-aware training, *leave-one-technique-out*: the model scored on technique *t* never saw *t* during training).

In [ ]:
X_, Y_, FAM_ = D["X"], D["y"], D["fam"]
_rng0 = np.random.default_rng(SEED0)
_ben = _rng0.permutation(np.where(Y_ == 0)[0]); _nb = len(_ben)
BEN_TE, BEN_VA, BEN_TR = _ben[:int(.2 * _nb)], _ben[int(.2 * _nb):int(.3 * _nb)], _ben[int(.3 * _nb):]

FAMS = D["info"]["families"]
_inter = [f.lower() for i in range(5) for c in FAMILIES for f in FAMILIES[c][i:i + 1]]   # interleave categories so a partial run covers all three
FOLD_ORDER = [f for f in _inter if f in FAMS] + [f for f in FAMS if f not in _inter]
FOLD_FAMS = FOLD_ORDER if LOFO_MAX_FOLDS is None else FOLD_ORDER[:LOFO_MAX_FOLDS]
CAT_OF = {f.lower(): c for c in FAMILIES for f in FAMILIES[c]}
CAT_OF.update({"cws": "Spyware", "coolwebsearch": "Spyware"})   # parse_family() shortens CoolWebSearch to "cws"
assert len(FAMS) >= 4, "LOFO needs >= 4 malware families"

def make_split(protocol, unit, seed):
    """Returns (train_idx, val_idx, test_idx, val_families)."""
    if protocol == "P1_random":
        idx = np.arange(len(Y_))
        tr_va, te = train_test_split(idx, test_size=0.2, stratify=Y_, random_state=seed)
        tr, va = train_test_split(tr_va, test_size=0.125, stratify=Y_[tr_va], random_state=seed)
        return tr, va, te, []
    others = [f for f in FAMS if f != unit]
    r = np.random.default_rng(SEED0 * 1000 + FAMS.index(unit))
    val_f = [str(f) for f in r.choice(others, size=min(LOFO_N_VAL_FAMILIES, len(others) - 2), replace=False)]
    tr_f = [f for f in others if f not in val_f]
    mal = np.where(Y_ == 1)[0]; pick = lambda S: mal[np.isin(FAM_[mal], list(S))]
    return np.r_[BEN_TR, pick(tr_f)], np.r_[BEN_VA, pick(val_f)], np.r_[BEN_TE, pick([unit])], val_f

class RuleBaseline:
    """Feature-space analogue of SPECTRE-style heuristic detectors: a rule fires when an anomaly indicator exceeds the
    99th percentile of benign training data; score = fraction of fired rules (voting). Not a re-implementation of SPECTRE."""
    def __init__(self, cols, prefixes=("malfind.", "psxview.", "ldrmodules."), q=0.99):
        self.idx = np.array([i for i, c in enumerate(cols) if c.startswith(prefixes)]); self.q = q
    def fit(self, Xb):
        self.thr = np.quantile(Xb[:, self.idx], self.q, axis=0); return self
    def score(self, X):
        return (X[:, self.idx] > self.thr).mean(1)

def _log1p(Z): return np.log1p(np.maximum(Z, 0.0))
def fit_lr(Xtr, ytr, seed):
    return make_pipeline(FunctionTransformer(_log1p, validate=False), StandardScaler(),
                         LogisticRegression(C=1.0, max_iter=3000, random_state=seed)).fit(Xtr, ytr)
def fit_rf(Xtr, ytr, seed):
    return RandomForestClassifier(n_estimators=RF_N_EST, min_samples_leaf=2, max_features="sqrt", n_jobs=-1, random_state=seed).fit(Xtr, ytr)
def fit_xgb(Xtr, ytr, Xva, yva, seed):
    m = xgb.XGBClassifier(n_estimators=XGB_N_EST, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                          reg_lambda=1.0, tree_method="hist", device=XGB_DEVICE, eval_metric="logloss",
                          early_stopping_rounds=30, random_state=seed, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False); return m
def prob(m): return lambda Z: m.predict_proba(Z)[:, 1]

def augment(Xtr, ytr, ref, techs, seed):
    """Evasion-aware training: add mimicry-perturbed (and feasibility-projected) copies of training malware (alpha ~ U(0.1, 1))."""
    rng = np.random.default_rng(seed + 1234); Xm = Xtr[ytr == 1]; parts, ys = [Xtr], [ytr]
    for t in techs:
        sel = rng.random(len(Xm)) < AUG_FRAC
        a = rng.uniform(0.1, 1.0, sel.sum())
        parts.append(evade(Xm[sel], TECH_IDX[t], ref, a, rng)); ys.append(np.ones(sel.sum(), dtype=int))
    return np.vstack(parts), np.concatenate(ys)

def _gap_center(s, tau):
    """Move tau to the middle of the gap between the next lower distinct score and tau (avoids an edge-of-class threshold)."""
    u = np.unique(s); lower = u[u < tau]
    return float((tau + lower[-1]) / 2) if len(lower) else float(tau)

def calibrate_thresholds(y, s, fpr_target=FPR_TARGET):
    fpr, tpr, thr = roc_curve(y, s); ok = np.isfinite(thr); J = tpr - fpr
    cand = thr[ok & (J >= J[ok].max() - 1e-12)]
    tau_j = _gap_center(s, float(np.median(cand)))
    m = ok & (fpr <= fpr_target)
    if m.any():
        i = np.argmax(np.where(m, tpr, -1)); tau_f = _gap_center(s, float(thr[i]))
    else:
        tau_f = float(np.max(s) + 1e-9)
    return {"Youden": tau_j, "FPR<=1%": tau_f}

def binary_metrics(y, s, thr):
    pred = (s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    dv = lambda a, b: a / b if b else 0.0
    prec, rec, spec = dv(tp, tp + fp), dv(tp, tp + fn), dv(tn, tn + fp)
    den = math.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return dict(auc=roc_auc_score(y, s) if len(set(y)) > 1 else np.nan,
                pr_auc=average_precision_score(y, s) if len(set(y)) > 1 else np.nan,
                precision=prec, recall=rec, specificity=spec, fpr=1 - spec,
                f1=dv(2 * prec * rec, prec + rec), acc=dv(tp + tn, len(y)),
                mcc=((tp * tn - fp * fn) / den) if den else 0.0, tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))

## 5. Model-aware adaptive attack
The mimicry attack above applies one fixed operation to a whole plugin group, so it cannot show that a detector *itself* is brittle. The **adaptive attack** gives the adversary black-box score access to the deployed detector and a budget of *k* edited features. Each edit sets one feature to its benign target value (the benign median) and the feasibility projection is re-applied (derived per-process / ratio features are recomputed and do not count against the budget).
* **Greedy**: for every row, try every unedited feature and keep the edit that lowers the detector score most; stop editing when no edit helps.
* **Random control**: the same budget, features chosen uniformly at random. The gap between the two shows how much a *model-aware* adversary gains over blind mimicry.
* The adversary keeps the best (lowest) score seen so far. A row counts as evaded at budget *k* when that score is below the frozen tau. **ASR@k** = share of initially detected malware rows evaded within *k* edits; we also report the median number of edits needed (rows never evaded count as `kmax + 1`).
* RF is not attacked (cost of ~10^5 forest evaluations per step); the rule baseline, LR, Model A and Model A-AT are.

In [ ]:
def run_attack(score_fn, Xm, ref, kmax, mode, rng, pool):
    """Returns running-min score trajectory, shape (kmax + 1, n)."""
    n, d = Xm.shape; cur = Xm.copy(); moved = np.zeros((n, d), bool); proj = ref.get("proj"); tgt = ref["target"]; ar = np.arange(n)
    s_cur = score_fn(cur); traj = np.empty((kmax + 1, n)); traj[0] = s_cur; run_min = s_cur.copy()
    def fix(Z, tm): return proj.project(Z, Xm, tm) if proj is not None else Z
    for k in range(1, kmax + 1):
        if mode == "greedy":
            S = np.full((n, d), np.inf); CAND = {}
            for j in pool:
                cand = cur.copy(); cand[:, j] = tgt[j]; tm = moved.copy(); tm[:, j] = True; cand = fix(cand, tm)
                S[:, j] = np.where(moved[:, j], np.inf, score_fn(cand)); CAND[j] = cand
            bj = S.argmin(1); bs = S[ar, bj]; improve = bs < s_cur - 1e-12
            if not improve.any(): traj[k:] = run_min[None, :]; break
            for j in pool:
                rows = improve & (bj == j)
                if rows.any(): cur[rows] = CAND[j][rows]; moved[rows, j] = True
            s_cur = np.where(improve, bs, s_cur)
        else:
            r = rng.random((n, len(pool))); r[moved[:, pool]] = -1.0; jj = np.asarray(pool)[r.argmax(1)]
            cur[ar, jj] = tgt[jj]; moved[ar, jj] = True; cur = fix(cur, moved); s_cur = score_fn(cur)
        run_min = np.minimum(run_min, s_cur); traj[k] = run_min
    return traj

def attack_rows(traj, tau, base, model, mode):
    """ASR@k among rows initially detected at tau (first = first budget k at which the running-min score falls below tau)."""
    det = traj[0] >= tau; out = []
    if det.sum() == 0: return out
    T = traj[:, det]; below = T < tau; first = np.where(below.any(0), below.argmax(0), traj.shape[0])   # traj.shape[0] = kmax + 1 -> never
    for k in ADAPT_KS:
        out.append(dict(base, model=model, attack=mode, k=k, asr=float((first <= k).mean()), n_attacked=int(det.sum()),
                        median_moves=float(np.median(first))))
    return out

## 6. Main experiment loop
For every run (15 LOFO folds + P1 seeds): fit the five detectors (and six A-AT variants), calibrate thresholds on the validation malware of unseen families, score the clean test data and every (technique x strength) evaded copy of the **test malware** with tau frozen. Also recorded per run: the feasibility ablation (projection on/off), jitter sensitivity, the adaptive attack, and (for Model B) the scores of every detector on the shared LLM sample.

In [ ]:
RUNS = []
if "P3_LOFO" in PROTOCOLS: RUNS += [dict(protocol="P3_LOFO", unit=f, seed=SEED0 + i) for i, f in enumerate(FOLD_FAMS)]
if "P1_random" in PROTOCOLS: RUNS += [dict(protocol="P1_random", unit=f"seed{SEED0 + s}", seed=SEED0 + s) for s in range(N_SEEDS_P1)]
ADAPT_MODELS = [M_AT, M_A, M_LR, M_RULE]
SENS_TECHS = ["Injection", "Cross-view", "Combined"]

# ---- shared LLM sample (built once; identical rows for every detector) -----------------------------------------------
LLM_SAMPLE = None
if RUN_LLM and "P3_LOFO" in PROTOCOLS:
    _r = np.random.default_rng(SEED0 + 7)
    _val_of = {r["unit"]: make_split("P3_LOFO", r["unit"], r["seed"])[3] for r in RUNS if r["protocol"] == "P3_LOFO"}
    _fams = sorted(set(FOLD_FAMS) | {f for v in _val_of.values() for f in v})
    _byf = {f: _r.permutation(np.where((Y_ == 1) & (FAM_ == f))[0]) for f in _fams}
    n_ = LLM_N_PER_FAMILY
    LLM_SAMPLE = dict(fams=_fams, val_of=_val_of,
                      T={f: _byf[f][:n_] for f in _fams},              # test rows (used when f is the held-out family)
                      C={f: _byf[f][n_:2 * n_] for f in _fams},        # calibration rows (used when f is a validation family), disjoint from T
                      B_te=_r.choice(BEN_TE, min(LLM_N_BENIGN, len(BEN_TE)), replace=False),
                      B_va=_r.choice(BEN_VA, min(LLM_N_BENIGN, len(BEN_VA)), replace=False))
    # Reference statistics from BENIGN TRAINING rows only (identical in every fold) -> no malware-family information in the evidence sheets.
    LLM_SAMPLE["ref"] = fit_reference(X_[BEN_TR], X_[BEN_TR], COLS)
    _ti = np.concatenate([LLM_SAMPLE["T"][f] for f in _fams]); _tf = np.concatenate([[f] * len(LLM_SAMPLE["T"][f]) for f in _fams])
    LLM_SAMPLE["T_idx"], LLM_SAMPLE["T_fam"] = _ti, _tf
    LLM_SAMPLE["EV"] = {c: evade(X_[_ti], TECH_IDX[c[0]], LLM_SAMPLE["ref"], c[1], np.random.default_rng([SEED0, TECHS.index(c[0]), int(c[1] * 1000)]))
                        for c in LLM_CONDITIONS}
    print(f"LLM sample: {len(_fams)} families x {n_} test rows, {len(LLM_SAMPLE['B_te'])} benign test rows, {len(LLM_CONDITIONS)} evaded conditions")

ROWS, AD_ROWS, ABL_ROWS, SENS_ROWS, FOLD_INFO, ART, PAYLOAD = [], [], [], [], [], {}, {}

def run_one(run):
    protocol, unit, seed = run["protocol"], run["unit"], run["seed"]; t0 = time.time()
    tr, va, te, val_f = make_split(protocol, unit, seed)
    Xtr, ytr, Xva, yva, Xte, yte = X_[tr], Y_[tr], X_[va], Y_[va], X_[te], Y_[te]
    ref = fit_reference(Xtr[ytr == 0], Xtr, COLS)
    rules = RuleBaseline(COLS).fit(Xtr[ytr == 0])
    A, LRm, RFm = fit_xgb(Xtr, ytr, Xva, yva, seed), fit_lr(Xtr, ytr, seed), fit_rf(Xtr, ytr, seed)
    mk = lambda f: dict(score=f, tau=calibrate_thresholds(yva, f(Xva)))
    M = {M_RULE: mk(rules.score), M_LR: mk(prob(LRm)), M_RF: mk(prob(RFm)), M_A: mk(prob(A))}
    AT = {}
    for held in SINGLE + ["ALL"]:
        used = SINGLE if held == "ALL" else [t for t in SINGLE if t != held]
        Xa, ya = augment(Xtr, ytr, ref, used, seed)
        m = fit_xgb(Xa, ya, Xva, yva, seed); AT[held] = dict(model=m, **mk(prob(m)))
    M[M_AT] = AT["ALL"]
    key = lambda name, tech: ("ALL" if tech in ("clean", "Combined") else tech) if name == M_AT else "-"
    mdl = lambda name, tech: AT[key(name, tech)] if name == M_AT else M[name]

    ben, mal = Xte[yte == 0], Xte[yte == 1]; yy = np.r_[np.zeros(len(ben), int), np.ones(len(mal), int)]
    t_l = time.time(); M[M_A]["score"](ben); lat_A = (time.time() - t_l) / max(len(ben), 1) * 1000
    base = dict(protocol=protocol, unit=unit); rows, cache, clean_scores = [], {}, {}
    def sben(name, tech):
        k = (name, key(name, tech))
        if k not in cache: cache[k] = mdl(name, tech)["score"](ben)
        return cache[k]
    for name in MODELS:
        m = mdl(name, "clean"); sb, sm = sben(name, "clean"), m["score"](mal); clean_scores[name] = (sb, sm)
        for kind in TAU_KINDS:
            rows.append(dict(base, model=name, tau_kind=kind, condition="clean", alpha=0.0, tau=m["tau"][kind], **binary_metrics(yy, np.r_[sb, sm], m["tau"][kind])))
    ev_comb = {}
    for tech in TECHS:
        for alpha in ALPHAS:
            Xev = evade(mal, TECH_IDX[tech], ref, alpha, np.random.default_rng([seed, TECHS.index(tech), int(alpha * 1000)]))
            for name in MODELS:
                m = mdl(name, tech); se = m["score"](Xev)
                if tech == "Combined" and alpha == PRIMARY_ALPHA: ev_comb[name] = se
                for kind in TAU_KINDS:
                    rows.append(dict(base, model=name, tau_kind=kind, condition=tech, alpha=alpha, tau=m["tau"][kind],
                                     **binary_metrics(yy, np.r_[sben(name, tech), se], m["tau"][kind])))
    # ---- feasibility ablation + jitter sensitivity (recall only; benign rows are untouched) -------------------------------
    abl, sens = [], []
    for tech in TECHS:
        for proj_on in ([True, False] if ref["proj"] is not None else [True]):
            Xev = evade(mal, TECH_IDX[tech], ref, PRIMARY_ALPHA, np.random.default_rng([seed, TECHS.index(tech), 7]), project=proj_on)
            viol = float(ref["proj"].violation(Xev).mean()) if ref["proj"] is not None else np.nan
            for name in MODELS:
                m = mdl(name, tech); se = m["score"](Xev)
                for kind in TAU_KINDS:
                    abl.append(dict(base, model=name, tau_kind=kind, condition=tech, projected=proj_on, viol_rate=viol, recall=float((se >= m["tau"][kind]).mean())))
    for tech in SENS_TECHS:
        for jit in SENS_JITTER:
            Xev = evade(mal, TECH_IDX[tech], ref, PRIMARY_ALPHA, np.random.default_rng([seed, TECHS.index(tech), 11]), jitter=jit)
            for name in MODELS:
                m = mdl(name, tech); se = m["score"](Xev)
                for kind in TAU_KINDS:
                    sens.append(dict(base, model=name, tau_kind=kind, condition=tech, jitter=jit, recall=float((se >= m["tau"][kind]).mean())))
    # ---- adaptive attack ---------------------------------------------------------------------------------------------------
    ad = []
    if ADAPT and len(mal):
        r_ = np.random.default_rng([seed, 99]); Xad = mal[r_.choice(len(mal), min(ADAPT_N, len(mal)), replace=False)]; pool = list(TECH_IDX["Combined"])
        for name in ADAPT_MODELS:
            m = mdl(name, "Combined")
            for mode in ("greedy", "random"):
                traj = run_attack(m["score"], Xad, ref, ADAPT_KMAX, mode, np.random.default_rng([seed, 5, len(mode)]), pool)
                for kind in TAU_KINDS: ad += attack_rows(traj, m["tau"][kind], dict(base, tau_kind=kind), name, mode)
    # ---- LLM payload: all detectors scored on the shared LLM sample (held-out family rows only) -------------------------------
    if protocol == "P3_LOFO" and LLM_SAMPLE is not None:
        S = LLM_SAMPLE; msk = S["T_fam"] == unit; Xt = X_[S["T"][unit]]
        pl = dict(val_fams=val_f, tau={n: M[n]["tau"] for n in MODELS}, benign={}, clean={}, ev={})
        for name in MODELS:
            pl["benign"][name] = mdl(name, "clean")["score"](X_[S["B_te"]]); pl["clean"][name] = mdl(name, "clean")["score"](Xt)
            for c in LLM_CONDITIONS:
                pl["ev"][(name, c)] = mdl(name, c[0])["score"](S["EV"][c][msk])
        PAYLOAD[unit] = pl
    art = dict(clean_scores=clean_scores, ev_comb=ev_comb, yva=yva, sva={n: M[n]["score"](Xva) for n in MODELS}, tau={n: M[n]["tau"] for n in MODELS},
               importance=A.get_booster().get_score(importance_type="gain"),
               feas=None if ref["proj"] is None else dict(report=ref["proj"].report, adopted=ref["proj"].adopted()))
    info = dict(base, seed=seed, n_train=len(tr), n_val=len(va), n_test_benign=int((yte == 0).sum()), n_test_malware=int((yte == 1).sum()),
                val_families=",".join(val_f), seconds=round(time.time() - t0, 1),
                category=CAT_OF.get(unit, "all") if protocol == "P3_LOFO" else "all", best_xgb_iters=int(getattr(A, "best_iteration", 0) or 0), ms_per_snapshot_A=lat_A)
    return rows, ad, abl, sens, art, info

t_all = time.time()
for i, run in enumerate(RUNS):
    rows, ad, abl, sens, art, info = run_one(run)
    ROWS += rows; AD_ROWS += ad; ABL_ROWS += abl; SENS_ROWS += sens; FOLD_INFO.append(info); ART[(run["protocol"], run["unit"])] = art
    el = time.time() - t_all
    print(f"[{i + 1:2d}/{len(RUNS)}] {run['protocol']:9s} {run['unit']:14s} {info['seconds']:6.1f}s | elapsed {el / 60:5.1f} min | ETA {el / (i + 1) * (len(RUNS) - i - 1) / 60:5.1f} min")
R, AD, ABL, SENS, FOLDS = map(pd.DataFrame, (ROWS, AD_ROWS, ABL_ROWS, SENS_ROWS, FOLD_INFO))
R.to_csv(OUT / "metrics_all_runs.csv", index=False); AD.to_csv(OUT / "adaptive_attack.csv", index=False)
ABL.to_csv(OUT / "feasibility_ablation.csv", index=False); SENS.to_csv(OUT / "jitter_sensitivity.csv", index=False); FOLDS.to_csv(OUT / "fold_info.csv", index=False)
print(f"done in {(time.time() - t_all) / 60:.1f} min; metric rows: {len(R)}")
_a = next(iter(ART.values()))["feas"]
if _a is not None:
    print("\nFeasibility constraints ADOPTED in the first run (verified on training rows):")
    print(f"  {len(_a['adopted']['leq'])} inequality, handle-sum: {_a['adopted']['handle_sum'] is not None}, {len(_a['adopted']['per_process'])} per-process averages, {len(_a['adopted']['ratio_groups'])} ratio groups")
    print("  Candidates NOT adopted (verified rate < threshold):", {k: round(v, 3) for k, v in _a["report"].items() if v < 0.9} or "none")

## 7. Evasion metrics and statistics
* **EIDD** (Evasion-Induced Detection Degradation) per technique *i*, at frozen tau: `EIDD_i = (Recall_clean - Recall_evaded_i) / Recall_clean x 100 %`.
* **Resilience Score** `R = 1 - mean_i(EIDD_i)/100` over the five single techniques (higher = more robust). **Worst-case recall** = minimum recall over the five singles and the Combined adversary; **mean evaded recall** = mean over the five singles.
* **Evasion Break Point (EBP50)**: smallest evasion strength `a` at which recall falls to half of its clean value (linear interpolation; `NaN` = never within a <= 1).
* **Unit of analysis = the fold** (one held-out family, or one P1 seed). Reported: mean, std and 95 % t-interval over folds. **Paired tests** compare two detectors on the *same* folds: two-sided Wilcoxon signed-rank on per-fold recall, **Holm-corrected** over the whole comparison family, plus a **cluster bootstrap** (resampling folds) 95 % CI of the mean difference. With 15 folds the smallest attainable Wilcoxon p is about 6e-5; with the 5-fold FAST preset it is 0.0625, so a significance claim needs the full run.

In [ ]:
def tci(x, conf=0.95):
    x = np.asarray(x, float); x = x[~np.isnan(x)]; n = len(x)
    if n == 0: return np.nan, np.nan, np.nan
    if n < 2: return x.mean(), np.nan, np.nan
    return x.mean(), x.std(ddof=1), stats.t.ppf((1 + conf) / 2, n - 1) * x.std(ddof=1) / np.sqrt(n)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty(len(p)); run = 0.0
    for r, i in enumerate(o): run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj

def boot_ci(v, B=2000, seed=0):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    if len(v) < 2: return np.nan, np.nan
    m = np.random.default_rng(seed).integers(0, len(v), (B, len(v))); b = v[m].mean(1)
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

KEY = ["protocol", "unit", "model", "tau_kind"]
clean = R[R.condition == "clean"][KEY + ["recall"]].rename(columns={"recall": "recall_clean"})
EV = R[R.condition != "clean"].merge(clean, on=KEY)
EV["eidd"] = np.where(EV.recall_clean > 0, (EV.recall_clean - EV.recall) / EV.recall_clean * 100, np.nan)

res = (EV[EV.condition.isin(SINGLE)].groupby(KEY + ["alpha"]).eidd.mean().reset_index().assign(resilience=lambda d: 1 - d.eidd / 100))
wc = EV.groupby(KEY + ["alpha"]).recall.min().rename("worst_recall").reset_index()
mer = EV[EV.condition.isin(SINGLE)].groupby(KEY + ["alpha"]).recall.mean().rename("mean_evaded_recall").reset_index()

def ebp(g):
    g = g.sort_values("alpha"); a = np.r_[0.0, g.alpha.values]; r = np.r_[g.recall_clean.iloc[0], g.recall.values]
    if r[0] <= 0: return np.nan
    tgt = 0.5 * r[0]
    for i in range(1, len(a)):
        if r[i] <= tgt: return a[i - 1] + (a[i] - a[i - 1]) * (r[i - 1] - tgt) / max(r[i - 1] - r[i], 1e-12)
    return np.nan
EBP = pd.DataFrame([dict(zip(KEY + ["condition"], k), ebp50=ebp(g)) for k, g in EV.groupby(KEY + ["condition"])])

def summarize(df, by, col):
    rows = []
    for k, g in df.groupby(by):
        m, sd, h = tci(g[col]); rows.append(dict(zip(by, k if isinstance(k, tuple) else (k,)), mean=m, std=sd, ci95=h, n=int(g[col].notna().sum())))
    return pd.DataFrame(rows)

S_eidd = summarize(EV, ["protocol", "model", "tau_kind", "condition", "alpha"], "eidd")
S_rec  = summarize(EV, ["protocol", "model", "tau_kind", "condition", "alpha"], "recall")
S_res  = summarize(res, ["protocol", "model", "tau_kind", "alpha"], "resilience")
S_wc   = summarize(wc, ["protocol", "model", "tau_kind", "alpha"], "worst_recall")
S_mer  = summarize(mer, ["protocol", "model", "tau_kind", "alpha"], "mean_evaded_recall")
S_ebp  = summarize(EBP, ["protocol", "model", "tau_kind", "condition"], "ebp50")
S_ebp["defined_fraction"] = [EBP[(EBP.protocol == r.protocol) & (EBP.model == r.model) & (EBP.tau_kind == r.tau_kind) & (EBP.condition == r.condition)].ebp50.notna().mean()
                             for r in S_ebp.itertuples()]
S_clean = summarize(R[R.condition == "clean"].melt(id_vars=KEY, value_vars=["auc", "pr_auc", "precision", "recall", "specificity", "fpr", "f1", "mcc", "acc"],
                                                   var_name="metric", value_name="v"), ["protocol", "model", "tau_kind", "metric"], "v")
for name, df in dict(summary_clean_metrics=S_clean, summary_eidd=S_eidd, summary_recall_evaded=S_rec, summary_resilience=S_res,
                     summary_worst_case_recall=S_wc, summary_mean_evaded_recall=S_mer, summary_ebp50=S_ebp).items():
    df.to_csv(OUT / f"{name}.csv", index=False)
TH = R[R.condition == "clean"].groupby(["protocol", "model", "tau_kind"]).tau.agg(["mean", "std"]).reset_index(); TH.to_csv(OUT / "thresholds.csv", index=False)

PRIM = PROTOCOLS[0]
def show(df, pivot_idx, pivot_col, val, kind="Youden", d=3, extra=None):
    g = df[(df.protocol == PRIM) & (df.tau_kind == kind)]
    if extra is not None: g = g[extra(g)]
    return g.pivot(index=pivot_idx, columns=pivot_col, values=val).reindex([m for m in MODELS if m in set(g[pivot_idx])]).round(d).to_string()
print(f"== Clean test performance ({PRIM}; Youden tau; mean over folds) ==")
print(show(S_clean, "model", "metric", "mean", extra=lambda g: g.metric.isin(["auc", "recall", "fpr", "f1", "mcc"])))
print(f"\n== Clean recall at FPR<=1% tau ==")
print(show(S_clean, "model", "metric", "mean", kind="FPR<=1%", extra=lambda g: g.metric.isin(["auc", "recall", "fpr", "f1"])))
print("\n== Resilience Score R (higher = better; Youden) ==");  print(show(S_res, "model", "alpha", "mean"))
print(f"\n== EIDD % at a = {PRIMARY_ALPHA} (Youden) ==");        print(show(S_eidd, "model", "condition", "mean", d=1, extra=lambda g: g.alpha == PRIMARY_ALPHA))
print(f"\n== Worst-case recall (Youden) ==");                     print(show(S_wc, "model", "alpha", "mean"))
print("\n== Evasion Break Point EBP50 (NaN = never reaches 50 % of clean recall) =="); print(show(S_ebp, "model", "condition", "mean", d=2))

In [ ]:
# --- Paired tests across folds: Wilcoxon + Holm + cluster bootstrap ------------------------------------------------
PAIRS = [(M_AT, M_A, "evasion-aware training vs plain XGBoost"), (M_AT, M_RF, "A-AT vs random forest"), (M_AT, M_LR, "A-AT vs logistic regression"),
         (M_A, M_RF, "XGBoost vs random forest"), (M_A, M_LR, "XGBoost vs logistic regression"), (M_A, M_RULE, "XGBoost vs rule baseline"),
         (M_RF, M_LR, "random forest vs logistic regression")]
def paired_table(prot, kind, alpha=PRIMARY_ALPHA):
    g = R[(R.protocol == prot) & (R.tau_kind == kind) & ((R.condition == "clean") | (R.alpha == alpha))]
    piv = g.pivot_table(index="unit", columns=["model", "condition"], values="recall"); rows = []
    for a_, b_, lab in PAIRS:
        for cond in ["clean"] + TECHS:
            if (a_, cond) not in piv.columns or (b_, cond) not in piv.columns: continue
            d = (piv[(a_, cond)] - piv[(b_, cond)]).dropna().values
            try: p = float(stats.wilcoxon(d).pvalue) if np.any(d != 0) else 1.0
            except ValueError: p = 1.0
            lo, hi = boot_ci(d)
            rows.append(dict(protocol=prot, tau_kind=kind, comparison=lab, model_a=a_, model_b=b_, condition=cond, alpha=0.0 if cond == "clean" else alpha,
                             mean_diff_recall=float(d.mean()) if len(d) else np.nan, boot_ci_lo=lo, boot_ci_hi=hi, n_folds=len(d),
                             folds_a_better=int((d > 0).sum()), folds_b_better=int((d < 0).sum()), p_wilcoxon=p))
    T = pd.DataFrame(rows)
    if len(T): T["p_holm"] = holm(T.p_wilcoxon.values)
    return T
PT = pd.concat([paired_table(pr, kd) for pr in PROTOCOLS for kd in TAU_KINDS], ignore_index=True)
PT.to_csv(OUT / "paired_tests.csv", index=False)
_s = PT[(PT.protocol == PRIM) & (PT.tau_kind == "Youden") & (PT.comparison.str.startswith("evasion-aware"))]
print(f"== Does evasion-aware training help? ({PRIM}, a = {PRIMARY_ALPHA}, Youden; recall difference A-AT minus A) ==")
print(_s[["condition", "mean_diff_recall", "boot_ci_lo", "boot_ci_hi", "folds_a_better", "folds_b_better", "p_wilcoxon", "p_holm"]].round(4).to_string(index=False))

# Cluster bootstrap CIs of the headline numbers (resampling FOLDS, not rows) --------------------------------------------
BC = []
for (pr, kd, nm), _g in R.groupby(["protocol", "tau_kind", "model"]):
    for lab, vec in (("clean recall", _g[_g.condition == "clean"].set_index("unit").recall),
                     (f"Combined recall a={PRIMARY_ALPHA}", _g[(_g.condition == "Combined") & (_g.alpha == PRIMARY_ALPHA)].set_index("unit").recall),
                     (f"Resilience a={PRIMARY_ALPHA}", res[(res.protocol == pr) & (res.tau_kind == kd) & (res.model == nm) & (res.alpha == PRIMARY_ALPHA)].set_index("unit").resilience)):
        lo, hi = boot_ci(vec.values); BC.append(dict(protocol=pr, tau_kind=kd, model=nm, quantity=lab, mean=float(vec.mean()), ci_lo=lo, ci_hi=hi, n_folds=len(vec)))
BC = pd.DataFrame(BC); BC.to_csv(OUT / "cluster_bootstrap_headline.csv", index=False)

# Per-family view (LOFO): which families are hardest to detect / most evasion-sensitive ----------------------------------
FAMT = pd.DataFrame()
if "P3_LOFO" in PROTOCOLS:
    c_ = R[(R.protocol == "P3_LOFO") & (R.tau_kind == "Youden") & (R.condition == "clean")].pivot(index="unit", columns="model", values="recall")
    e_ = R[(R.protocol == "P3_LOFO") & (R.tau_kind == "Youden") & (R.condition == "Combined") & (R.alpha == PRIMARY_ALPHA)].pivot(index="unit", columns="model", values="recall")
    FAMT = pd.concat({"clean": c_, f"Combined a={PRIMARY_ALPHA}": e_}, axis=1); FAMT.insert(0, "category", [CAT_OF.get(u, "") for u in FAMT.index])
    FAMT.to_csv(OUT / "per_family_recall_lofo.csv"); print("\n== Per-family recall, Model A / A-AT (LOFO, Youden) =="); print(FAMT[[("category", "")] + [(k, m) for k in ("clean", f"Combined a={PRIMARY_ALPHA}") for m in (M_A, M_AT)]].round(3).to_string())

In [ ]:
# --- Adaptive attack, feasibility ablation, jitter sensitivity summaries -------------------------------------------------
S_ad = pd.DataFrame(); S_mv = pd.DataFrame()
if len(AD):
    S_ad = summarize(AD, ["protocol", "model", "tau_kind", "attack", "k"], "asr"); S_ad.to_csv(OUT / "summary_adaptive_asr.csv", index=False)
    S_mv = summarize(AD.drop_duplicates(["protocol", "unit", "model", "tau_kind", "attack"]), ["protocol", "model", "tau_kind", "attack"], "median_moves"); S_mv.to_csv(OUT / "summary_adaptive_moves.csv", index=False)
    kk = [k for k in (4, 8) if k in ADAPT_KS] or ADAPT_KS[-1:]
    print(f"== Adaptive attack ({PRIM}, FPR<=1% tau): ASR@k, greedy vs random; median edits needed ==")
    g = S_ad[(S_ad.protocol == PRIM) & (S_ad.tau_kind == "FPR<=1%") & S_ad.k.isin(kk)].pivot_table(index="model", columns=["attack", "k"], values="mean")
    mv = S_mv[(S_mv.protocol == PRIM) & (S_mv.tau_kind == "FPR<=1%")].pivot(index="model", columns="attack", values="mean").add_prefix("median edits ")
    g.columns = [f"ASR@{k} {a}" for a, k in g.columns]
    print(g.join(mv).reindex([m for m in MODELS if m in g.index]).round(2).to_string())
if len(ABL):
    S_abl = ABL.groupby(["protocol", "model", "tau_kind", "condition", "projected"])[["recall", "viol_rate"]].mean().reset_index(); S_abl.to_csv(OUT / "summary_feasibility_ablation.csv", index=False)
    g = ABL[(ABL.protocol == PRIM) & (ABL.tau_kind == "FPR<=1%") & (ABL.model == M_A)].groupby(["condition", "projected"])[["recall", "viol_rate"]].mean().unstack("projected")
    print(f"\n== Feasibility ablation ({M_A}, a = {PRIMARY_ALPHA}, FPR<=1% tau): recall and share of rows breaking an adopted constraint ==")
    print(g.round(3).to_string())
if len(SENS):
    S_sens = summarize(SENS, ["protocol", "model", "tau_kind", "condition", "jitter"], "recall"); S_sens.to_csv(OUT / "summary_jitter_sensitivity.csv", index=False)

# --- Paper-ready tables (LaTeX + CSV) -------------------------------------------------------------------------------------
def pm(df, model, d=3):
    r = df[df.model == model]; return f"{r['mean'].iloc[0]:.{d}f} $\\pm$ {r['std'].iloc[0]:.{d}f}" if len(r) and not np.isnan(r['std'].iloc[0]) else (f"{r['mean'].iloc[0]:.{d}f}" if len(r) else "n/a")
def table_main(kind):
    sc = S_clean[(S_clean.protocol == PRIM) & (S_clean.tau_kind == kind)]; ev = lambda df, **kw: df[(df.protocol == PRIM) & (df.tau_kind == kind) & np.all([df[k] == v for k, v in kw.items()], axis=0)]
    rows = []
    for name in MODELS:
        g = lambda met: pm(sc[sc.metric == met], name)
        rows.append([name, g("auc"), g("f1"), g("recall"), g("fpr"), pm(ev(S_rec, condition="Combined", alpha=PRIMARY_ALPHA), name),
                     pm(ev(S_wc, alpha=PRIMARY_ALPHA), name), pm(ev(S_res, alpha=PRIMARY_ALPHA), name)])
    return pd.DataFrame(rows, columns=["Model", "AUC", "F1", "Recall", "FPR", f"Recall (Comb., a={PRIMARY_ALPHA})", f"Worst-case recall", "Resilience R"])
def to_latex(df, caption, label):
    body = " \\\\\n".join(" & ".join(map(str, r)) for r in df.values)
    return ("\\begin{table*}[t]\n\\centering\n\\caption{%s}\n\\label{%s}\n\\footnotesize\n\\begin{tabular}{l%s}\n\\hline\n%s \\\\\n\\hline\n%s \\\\\n\\hline\n\\end{tabular}\n\\end{table*}\n"
            % (caption, label, "c" * (df.shape[1] - 1), " & ".join(df.columns), body))
T1, T2 = table_main("Youden"), table_main("FPR<=1%")
for nm, T, kd, lab in (("table1_main_results_youden", T1, "Youden", "tab:main"), ("table2_main_results_fpr1", T2, "FPR$\\leq$1\\%", "tab:main_fpr")):
    T.to_csv(OUT / f"{nm}.csv", index=False)
    (OUT / f"{nm}.tex").write_text(to_latex(T, f"Clean performance and robustness, leave-one-family-out ({kd} threshold, mean $\\pm$ std over folds; evasion strength a = {PRIMARY_ALPHA}, feasibility-projected mimicry).", lab))
print("\n== Table I (Youden) =="); print(T1.to_string(index=False)); print("\n== Table II (FPR<=1%) =="); print(T2.to_string(index=False))

## 8. Publication figures (IEEE style)
Single-column width = 3.5 in, double-column = 7.16 in. **Design rules:** no shaded confidence bands and no error bars inside the plots (fold-to-fold spread is in the tables / CSV files, and per-family detail is in Fig. 6); values are printed on the bars instead; bars within a group touch, with a clear gap only between groups. Colour is never the only encoding (marker, line style and hatch differ per model). The **rule baseline is drawn only where it adds information** (Fig. 1, 6, 8): with AUC ~0.55 under leave-one-family-out it is not a usable detector, and its "evaded recall" can *rise* (negative EIDD, R > 1), which is a property of a near-random score, not of robustness. Each figure is saved as 300-dpi PNG and as vector PDF with embedded TrueType fonts.

In [ ]:
def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight"); fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.show() if "get_ipython" in globals() else plt.close(fig); print("saved", name)

LBL = {M_AT: "Model A-AT", M_A: "Model A", M_RF: "Random forest", M_LR: "Logistic reg.", M_RULE: "Rule baseline", M_B: "Model B (LLM)"}
PLOT_MODELS = [M_AT, M_A, M_RF, M_LR]                      # learned detectors
UNITS = [u for (p, u) in ART if p == PRIM]; N_U = len(UNITS)
def legend_top(fig, handles, labels, ncol, y=1.03): fig.legend(handles, labels, loc="upper center", ncol=ncol, frameon=False, bbox_to_anchor=(0.5, y), fontsize=6.5, columnspacing=1.2, handlelength=2.2)
def model_handles(models, bar=False):
    import matplotlib.patches as mp, matplotlib.lines as ml
    if bar: return [mp.Patch(fc=STYLE[m]["color"], ec="black", hatch=STYLE[m]["hatch"], lw=0.4) for m in models], [LBL[m] for m in models]
    return [ml.Line2D([], [], color=STYLE[m]["color"], ls=STYLE[m]["ls"], marker=STYLE[m]["marker"], mec="black", mew=0.4) for m in models], [LBL[m] for m in models]
def grouped_bars(ax, models, groups, val, w=0.2, fmt="{:.2f}", label_min=None, fs=5.2):
    """val(model, group) -> float or nan. Bars of one group touch; groups are 1.0 apart."""
    n = len(models); x0 = np.arange(len(groups))
    for i, m in enumerate(models):
        st = STYLE[m]; v = np.array([val(m, g) for g in groups], float); xs = x0 + (i - (n - 1) / 2) * w
        ax.bar(xs, np.nan_to_num(v), w, color=st["color"], edgecolor="black", linewidth=0.4, hatch=st["hatch"])
        for xx, vv in zip(xs, v):
            if np.isfinite(vv) and (label_min is None or abs(vv) >= label_min): ax.text(xx, vv, fmt.format(vv), ha="center", va="bottom" if vv >= 0 else "top", fontsize=fs, rotation=90)
    ax.set_xticks(x0); ax.set_xticklabels(groups); ax.set_xlim(-0.55, len(groups) - 0.45)

# ==== ART-DEPENDENT FIGURES (need per-fold score arrays; skipped when re-drawing from CSV files) ====
if ART:
    gF = np.logspace(-4, 0, 400)
    # Fig 1 - fold-averaged ROC: full range, and zoom on the learned detectors -------------------------------------------------
    fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.7))
    for name in MODELS:
        T_ = []
        for u in UNITS:
            sb, sm = ART[(PRIM, u)]["clean_scores"][name]; f, t, _ = roc_curve(np.r_[np.zeros(len(sb)), np.ones(len(sm))], np.r_[sb, sm]); T_.append(np.interp(gF, f, t))
        auc_m = S_clean[(S_clean.protocol == PRIM) & (S_clean.tau_kind == "Youden") & (S_clean.model == name) & (S_clean.metric == "auc")]["mean"].iloc[0]; st = STYLE[name]
        for a_ in ax: a_.plot(gF, np.mean(T_, 0), color=st["color"], ls=st["ls"], lw=1.3, label=f"{LBL[name]} (AUC {auc_m:.4f})")
    ax[0].set_ylim(0, 1.02); ax[0].set_title(f"(a) ROC, clean, unseen family (mean of {N_U} folds)"); ax[1].set_ylim(0.9, 1.002); ax[1].set_title("(b) Zoom on the learned detectors")
    for a_ in ax: a_.set_xscale("log"); a_.set_xlim(1e-4, 1); a_.set_xlabel("False positive rate (log)")
    ax[0].set_ylabel("True positive rate"); ax[0].legend(loc="center left", frameon=False, fontsize=6, bbox_to_anchor=(0.0, 0.45))
    fig.tight_layout(); savefig(fig, "fig1_roc")

    # Fig 5 - why the threshold must be frozen: TPR/FPR vs tau, clean vs evaded (Model A, mean over folds) -------------------------------
    grid = np.linspace(0, 1, 301); fp_, tc_, te_ = [], [], []
    for u in UNITS:
        a_ = ART[(PRIM, u)]; sb, sm = a_["clean_scores"][M_A]; se = a_["ev_comb"][M_A]
        fp_.append([(sb >= t).mean() for t in grid]); tc_.append([(sm >= t).mean() for t in grid]); te_.append([(se >= t).mean() for t in grid])
    tauJ = np.mean([ART[(PRIM, u)]["tau"][M_A]["Youden"] for u in UNITS]); tau1 = np.mean([ART[(PRIM, u)]["tau"][M_A]["FPR<=1%"] for u in UNITS])
    fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.7)); c = STYLE[M_A]["color"]
    ax[0].plot(grid, np.mean(tc_, 0), color=c, label="TPR, clean malware"); ax[0].plot(grid, np.mean(te_, 0), color="black", ls="--", label=f"TPR, Combined evasion a={PRIMARY_ALPHA}")
    ax[0].plot(grid, np.mean(fp_, 0), color=STYLE[M_RULE]["color"], ls=":", label="FPR, benign")
    ax[0].axvline(tauJ, color="grey", lw=0.8); ax[0].text(tauJ + 0.01, 0.52, "frozen $\\tau$", fontsize=6, rotation=90, color="grey")
    ax[0].set_xlabel("Decision threshold $\\tau$"); ax[0].set_ylabel("Rate (mean over folds)"); ax[0].set_ylim(-0.02, 1.05); ax[0].set_title("(a) A frozen $\\tau$ keeps FPR low but loses the evaded malware")
    ax[0].legend(frameon=False, fontsize=6, loc="center right")
    u0 = UNITS[0]; sb, sm = ART[(PRIM, u0)]["clean_scores"][M_A]; se = ART[(PRIM, u0)]["ev_comb"][M_A]; bins = np.linspace(0, 1, 41)
    ax[1].hist(sb, bins, color="#2a78d6", label="Benign", edgecolor="white", lw=0.2); ax[1].hist(sm, bins, color="#eb6834", label="Held-out malware, clean", edgecolor="white", lw=0.2)
    ax[1].hist(se, bins, histtype="step", color="black", lw=1.0, label=f"Same malware, Combined a={PRIMARY_ALPHA}")
    ax[1].axvline(ART[(PRIM, u0)]["tau"][M_A]["Youden"], color="grey", lw=0.8); ax[1].set_yscale("log"); ax[1].set_xlabel("Model A score"); ax[1].set_ylabel("Count (log)")
    ax[1].set_ylim(0.7, ax[1].get_ylim()[1] * 40); ax[1].legend(frameon=False, fontsize=6, loc="upper center"); ax[1].set_title(f"(b) Test scores, fold: {u0} held out")
    fig.tight_layout(); savefig(fig, "fig5_threshold_scores")

    # Fig 7 - feature importance (share of total gain) -------------------------------------------------------------------------------------
    _imp = pd.DataFrame([{COLS[int(k[1:])] if k.startswith("f") and k[1:].isdigit() else k: v for k, v in ART[(PRIM, u)]["importance"].items()} for u in UNITS]).fillna(0.0)
    _imp = _imp.div(_imp.sum(1), axis=0).mean() * 100; imp = _imp.sort_values().tail(10)
    fig, ax = plt.subplots(figsize=(3.5, 2.6)); ax.barh(imp.index, imp.values, color=STYLE[M_A]["color"], edgecolor="black", linewidth=0.4, height=0.7)
    for y_, v in enumerate(imp.values): ax.text(v + 0.8, y_, f"{v:.1f}", va="center", fontsize=6)
    ax.set_xlabel("Share of total gain (%), mean over folds"); ax.set_title("Top-10 features, Model A"); ax.grid(axis="y", alpha=0); ax.set_xlim(0, imp.max() * 1.15)
    fig.tight_layout(); savefig(fig, "fig7_feature_importance")

# ==== FRAME-BASED FIGURES (only need the summary tables, so they can be re-drawn from the CSV files) ====
def _s(df, **kw):
    m = (df.protocol == PRIM)
    for k, v in kw.items(): m &= (df[k] == v)
    return df[m]
def _v(df, model, col="mean", **kw):
    r = _s(df, model=model, **kw)[col]; return float(r.iloc[0]) if len(r) else np.nan
KIND = "Youden"
easy = [t for t in SINGLE if all(_v(S_rec, m, tau_kind=KIND, condition=t, alpha=a) >= 0.95 for m in PLOT_MODELS for a in ALPHAS)]
hard = [t for t in TECHS if t not in easy]

# Fig 2 - Recall vs evasion strength ------------------------------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(hard), figsize=(7.16, 2.55), sharey=True); xs = [0.0] + ALPHAS
for ax_, tech in zip(np.atleast_1d(axes), hard):
    for name in PLOT_MODELS:
        y_ = [_v(S_clean, name, tau_kind=KIND, metric="recall")] + [_v(S_rec, name, tau_kind=KIND, condition=tech, alpha=a) for a in ALPHAS]; st = STYLE[name]
        ax_.plot(xs, y_, color=st["color"], ls=st["ls"], marker=st["marker"], mec="black", mew=0.4, ms=3.6)
    ax_.set_title(tech); ax_.set_ylim(-0.02, 1.04); ax_.set_xlim(-0.04, 1.04); ax_.axhline(0.5, color="grey", lw=0.5, ls=":"); ax_.set_xlabel("Evasion strength a")
np.atleast_1d(axes)[0].set_ylabel("Recall at frozen $\\tau$ (mean of folds)")
if easy:
    lo = min(_v(S_rec, m, tau_kind=KIND, condition=t, alpha=a) for m in PLOT_MODELS for t in easy for a in ALPHAS)
    fig.text(0.5, -0.03, f"Not drawn: {', '.join(easy)}. Every learned detector keeps recall $\\geq$ {lo:.2f} at every strength a $\\leq$ 1 for these techniques.", ha="center", fontsize=6.3)
legend_top(fig, *model_handles(PLOT_MODELS), ncol=4); fig.tight_layout(); savefig(fig, "fig2_recall_vs_evasion")

# Fig 3 - EIDD -------------------------------------------------------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.16, 2.7))
grouped_bars(ax, PLOT_MODELS, TECHS, lambda m, t: _v(S_eidd, m, tau_kind=KIND, condition=t, alpha=PRIMARY_ALPHA), w=0.2, fmt="{:.0f}", label_min=1.0)
ax.set_ylabel("EIDD (%), lower is better"); ax.set_ylim(0, 118); ax.set_title(f"Evasion-induced detection degradation at a = {PRIMARY_ALPHA} (mean of folds; unlabelled bars < 1 %)")
legend_top(fig, *model_handles(PLOT_MODELS, True), ncol=4, y=1.0); fig.tight_layout(rect=(0, 0, 1, 0.93)); savefig(fig, "fig3_eidd")

# Fig 4 - Resilience, worst-case recall, Evasion Break Point ------------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(7.16, 2.8), gridspec_kw=dict(width_ratios=[1, 1, 1.9]))
als = [PRIMARY_ALPHA, 1.0]; names = [f"a = {PRIMARY_ALPHA}", "a = 1.0"]
for k, (S_, ttl, yl) in enumerate([(S_res, "(a) Resilience Score R", "R (higher = better)"), (S_wc, "(b) Worst-case recall", "min recall over 6 conditions")]):
    grouped_bars(ax[k], PLOT_MODELS, als, lambda m, a, S_=S_: _v(S_, m, tau_kind=KIND, alpha=a), w=0.2, fmt="{:.2f}", label_min=0.005)
    ax[k].set_xticklabels(names); ax[k].set_ylim(0, 1.12); ax[k].set_ylabel(yl); ax[k].set_title(ttl)
grouped_bars(ax[2], PLOT_MODELS, TECHS, lambda m, t: _v(S_ebp, m, tau_kind=KIND, condition=t), w=0.2, fmt="{:.2f}", fs=5)
for j, t in enumerate(TECHS):                       # techniques where the break point was never reached: say so instead of drawing a fake bar
    if all(np.isnan(_v(S_ebp, m, tau_kind=KIND, condition=t)) for m in PLOT_MODELS): ax[2].text(j, 0.03, "never\nreached", ha="center", fontsize=5.5, color="grey")
ax[2].set_xticklabels(TECHS, rotation=30, ha="right"); ax[2].set_ylim(0, 1.12); ax[2].set_ylabel("EBP$_{50}$ (evasion strength a)"); ax[2].set_title("(c) Evasion Break Point")
legend_top(fig, *model_handles(PLOT_MODELS, True), ncol=4, y=1.02); fig.tight_layout(rect=(0, 0, 1, 0.93)); savefig(fig, "fig4_resilience_ebp")

# Fig 6 - Per-family recall heat map (LOFO) ---------------------------------------------------------------------------------------------------
if len(FAMT):
    HM = [M_A, M_AT, M_RF, M_LR]; order = sorted(FAMT.index, key=lambda u: (FAMT.loc[u, ("category", "")], u)); cols_ = [(c, m) for c in ("clean", f"Combined a={PRIMARY_ALPHA}") for m in HM]
    Z = FAMT.loc[order, cols_].values.astype(float); cats = [FAMT.loc[u, ("category", "")] for u in order]
    fig, ax = plt.subplots(figsize=(5.6, 0.25 * len(order) + 1.3)); im = ax.imshow(Z, cmap="Blues", vmin=0, vmax=1, aspect="auto"); ax.grid(False)
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]): ax.text(j, i, f"{Z[i, j]:.2f}", ha="center", va="center", fontsize=5.5, color="white" if Z[i, j] > 0.6 else "black")
    ax.set_xticks(range(len(cols_))); ax.set_xticklabels([LBL[m] for _, m in cols_], rotation=40, ha="right", fontsize=6)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([f"{u}" for u in order], fontsize=6)
    ax.axvline(len(HM) - 0.5, color="black", lw=1.2)
    for i in range(1, len(order)):
        if cats[i] != cats[i - 1]: ax.axhline(i - 0.5, color="black", lw=0.8)
    for c in dict.fromkeys(cats): ax.text(len(cols_) - 0.35, np.mean([i for i, x in enumerate(cats) if x == c]), c, rotation=-90, va="center", ha="left", fontsize=6.5)
    for xc, t in (((len(HM) - 1) / 2, "Clean"), (len(HM) + (len(HM) - 1) / 2, f"Combined evasion, a = {PRIMARY_ALPHA}")): ax.text(xc, -0.75, t, ha="center", va="bottom", fontsize=7)
    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.09); cb.set_label("Recall (Youden $\\tau$)", fontsize=7); cb.ax.tick_params(labelsize=6)
    fig.tight_layout(); savefig(fig, "fig6_per_family_recall")

# Fig 8 - Protocol effect (leakage / ceiling check) ----------------------------------------------------------------------------------------
if len(PROTOCOLS) > 1:
    fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.7)); PM = [M_AT, M_A, M_RF, M_LR, M_RULE]; PN = {"P3_LOFO": "P3 leave-one-family-out", "P1_random": "P1 random split"}
    for k, (metric, ttl, kind) in enumerate([("f1", "(a) F1 at Youden $\\tau$", "Youden"), ("recall", "(b) Recall at FPR$\\leq$1 % $\\tau$", "FPR<=1%")]):
        for j, pr in enumerate(PROTOCOLS):
            v = [float(S_clean[(S_clean.protocol == pr) & (S_clean.tau_kind == kind) & (S_clean.metric == metric) & (S_clean.model == m)]["mean"].iloc[0]) for m in PM]
            xs = np.arange(len(PM)) + (j - 0.5) * 0.38; ax[k].bar(xs, v, 0.38, color=["#2a78d6", "#eb6834"][j], edgecolor="black", linewidth=0.4, hatch=["", "//"][j], label=PN.get(pr, pr))
            for xx, vv in zip(xs, v): ax[k].text(xx, vv + 0.01, f"{vv:.2f}", ha="center", fontsize=5, rotation=90)
        ax[k].set_xticks(range(len(PM))); ax[k].set_xticklabels([LBL[m] for m in PM], rotation=25, ha="right"); ax[k].set_ylim(0, 1.2); ax[k].set_title(ttl); ax[k].set_xlim(-0.6, len(PM) - 0.4)
    h, l = ax[0].get_legend_handles_labels(); legend_top(fig, h, l, ncol=2, y=1.02); fig.tight_layout(rect=(0, 0, 1, 0.93)); savefig(fig, "fig8_protocol_effect")

# Fig 9 - Adaptive attack ----------------------------------------------------------------------------------------------------------------------
if len(S_ad):
    AM = [m for m in ADAPT_MODELS if m != M_RULE]; fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.8), gridspec_kw=dict(width_ratios=[1.5, 1])); ks = [0] + list(ADAPT_KS)
    for name in AM:
        st = STYLE[name]
        for mode in ("greedy", "random"):
            g = _s(S_ad, tau_kind="FPR<=1%", model=name, attack=mode).sort_values("k")
            if len(g): ax[0].plot(ks, np.r_[0.0, g["mean"].values], color=st["color"], ls=st["ls"] if mode == "greedy" else (0, (1, 2)), lw=1.3 if mode == "greedy" else 0.9,
                                  marker=st["marker"] if mode == "greedy" else None, mec="black", mew=0.4, ms=3.6, alpha=1.0 if mode == "greedy" else 0.6)
    ax[0].set_xlabel("Edit budget k (features set to the benign median)"); ax[0].set_ylabel("Attack success rate ASR@k"); ax[0].set_ylim(-0.02, 1.04); ax[0].set_title("(a) Greedy attacker (solid) vs random control (dotted)")
    mv = _s(S_mv, tau_kind="FPR<=1%", attack="greedy").set_index("model")
    ax[1].bar(range(len(AM)), [mv.loc[m, "mean"] for m in AM], 0.62, color=[STYLE[m]["color"] for m in AM], edgecolor="black", linewidth=0.4, hatch=None)
    for i, m in enumerate(AM): ax[1].bar(i, mv.loc[m, "mean"], 0.62, fc="none", ec="black", lw=0.4, hatch=STYLE[m]["hatch"]); ax[1].text(i, mv.loc[m, "mean"] + 0.1, f"{mv.loc[m, 'mean']:.1f}", ha="center", fontsize=6.5)
    ax[1].set_xticks(range(len(AM))); ax[1].set_xticklabels([LBL[m] for m in AM], rotation=20, ha="right"); ax[1].set_ylim(0, mv.loc[AM, "mean"].max() * 1.18); ax[1].set_ylabel("Median edits needed to evade"); ax[1].set_title("(b) Cost of evasion (greedy)")
    legend_top(fig, *model_handles(AM), ncol=3, y=1.02); fig.tight_layout(rect=(0, 0, 1, 0.92)); savefig(fig, "fig9_adaptive_attack")

# Fig 10 - Feasibility ablation and adversary-noise sensitivity ----------------------------------------------------------------------------------
if len(ABL) and len(S_sens):
    fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.8)); g = ABL[(ABL.protocol == PRIM) & (ABL.tau_kind == "FPR<=1%") & (ABL.model == M_A)].groupby(["condition", "projected"])[["recall", "viol_rate"]].mean().unstack("projected").loc[TECHS]
    x = np.arange(len(TECHS)); w = 0.36
    ax[0].bar(x - w / 2, g[("recall", True)], w, color="#2a78d6", edgecolor="black", linewidth=0.4, label="with feasibility projection")
    ax[0].bar(x + w / 2, g[("recall", False)], w, color="#eb6834", edgecolor="black", linewidth=0.4, hatch="//", label="unconstrained")
    for i, v in enumerate(g[("viol_rate", False)].values):
        if v >= 0.02: ax[0].text(i + w / 2, g[("recall", False)].values[i] + 0.02, f"{v * 100:.0f}% of rows\ninfeasible", ha="center", fontsize=5.2)
    ax[0].set_xticks(x); ax[0].set_xticklabels(TECHS, rotation=25, ha="right"); ax[0].set_ylim(0, 1.45); ax[0].set_xlim(-0.6, len(TECHS) - 0.4); ax[0].set_ylabel(f"Model A recall at a = {PRIMARY_ALPHA}")
    ax[0].set_title("(a) Feasibility ablation"); ax[0].legend(frameon=False, fontsize=6, loc="upper left", ncol=2)
    for name in PLOT_MODELS:
        st = STYLE[name]; gg = _s(S_sens, tau_kind="FPR<=1%", model=name, condition="Combined").sort_values("jitter")
        ax[1].plot(gg.jitter.values, gg["mean"].values, color=st["color"], ls=st["ls"], marker=st["marker"], mec="black", mew=0.4, ms=3.6)
    ax[1].set_xlabel("Adversary noise (fraction of benign std)"); ax[1].set_ylabel(f"Recall, Combined a = {PRIMARY_ALPHA}"); ax[1].set_ylim(0, 0.6); ax[1].set_title("(b) Sensitivity to adversary noise")
    legend_top(fig, *model_handles(PLOT_MODELS), ncol=4, y=1.02); fig.tight_layout(rect=(0, 0, 1, 0.92)); savefig(fig, "fig10_feasibility_noise")

# Fig 11 - Model B (LLM): reliability + comparison under evasion (called from the LLM cell or the CSV re-draw script) ---------------------------------
def make_fig11(S_llm, y_p, pB, pBp, pA, SFX=""):
    LM = [M_AT, M_A, M_RF, M_LR, M_B]; conds = [("clean", 0.0)] + list(LLM_CONDITIONS); labs = ["clean"] + [f"{t}\na={a}" for t, a in LLM_CONDITIONS]
    fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.9))
    for lab, p_, st in ((M_A, pA, STYLE[M_A]), (M_B + " raw", pB, STYLE[M_B]), (M_B + " + Platt", pBp, dict(color="#1baf7a", marker="^", ls="-"))):
        b = np.digitize(p_, np.linspace(0, 1, 11)[1:-1]); xs_, ys_ = [], []
        for k in range(10):
            m = b == k
            if m.sum() >= 5: xs_.append(p_[m].mean()); ys_.append(y_p[m].mean())
        ax[0].plot(xs_, ys_, color=st["color"], marker=st["marker"], mec="black", mew=0.4, ms=3.6, ls=st["ls"], label=f"{lab} (ECE {ece_score(y_p, p_):.3f})")
    ax[0].plot([0, 1], [0, 1], color="grey", lw=0.6, ls=":"); ax[0].set_xlabel("Predicted P(malicious)", labelpad=2); ax[0].set_ylabel("Observed fraction malicious"); ax[0].set_xlim(-0.03, 1.03); ax[0].set_ylim(-0.03, 1.03)
    ax[0].set_title("(a) Reliability, pooled folds" + (" [MOCK]" if SFX else "")); ax[0].legend(frameon=False, fontsize=5.8, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=1)
    grouped_bars(ax[1], LM, labs, lambda m, c: _v_llm(S_llm, m, conds[labs.index(c)]), w=0.17, fmt="{:.2f}", fs=5, label_min=0.005)
    ax[1].set_xticklabels(labs, fontsize=6); ax[1].set_ylim(0, 1.22); ax[1].set_ylabel("Recall at frozen $\\tau$"); ax[1].set_title("(b) Same test rows, all detectors" + (" [MOCK]" if SFX else ""))
    legend_top(fig, *model_handles(LM, True), ncol=5, y=1.02); fig.tight_layout(rect=(0, 0, 1, 0.93)); savefig(fig, f"fig11_llm{SFX}")
def _v_llm(S, m, c):
    r = S[(S.model == m) & (S.tau_kind == "Youden") & (S.condition == c[0]) & (S.alpha == c[1])]["mean"]; return float(r.iloc[0]) if len(r) else np.nan

## 9. Model B - LLM layer with structured-evidence prompting (optional, exploratory)
**Design.** The LLM never sees raw dumps. For each snapshot we build a short *evidence sheet*: a fixed list of `LLM_INDICATORS`, each printed with its value, the benign reference median / 5th-95th percentile and a status flag (ABOVE / BELOW / within the benign range), plus a one-line glossary of the Volatility plugins involved.
**Leakage control.** (i) The indicator list is chosen from forensic domain knowledge, *not* from model importances, so no held-out-family information selects it. (ii) The benign reference statistics come from the benign **training** rows only (identical in every fold). (iii) The LLM is zero-shot: no weights are trained on any malware family. (iv) The threshold and Platt calibration of the LLM in fold *f* use only a clean **validation** sample of benign rows plus the *validation families of that fold* (never family *f*).
**Scoring.** Instead of free-text parsing we read the next-token logits of the two answer words: `p(malicious) = sigmoid(logit(MALICIOUS) - logit(BENIGN))`. One forward pass per snapshot, no parsing failures, and a continuous score so ROC / thresholds / calibration all work.
**Protocol.** Per fold the held-out family contributes `LLM_N_PER_FAMILY` test rows (clean + evaded copies) plus a shared benign test sample. Model A / A-AT / RF / LR / rules (trained in that fold, thresholds from that fold) are scored on the **identical rows**. Metrics are computed per fold and averaged over folds, like everything else.
**Runtime.** Needs a GPU (Kaggle T4) and the model weights (Internet ON, or attach the model via *Add Input -> Models*). If the weights cannot be loaded the cell stops with a clear message; set `RUN_LLM = False` to skip. Never report the `[MOCK]` outputs.

In [ ]:
import importlib.util
LLM_INDICATORS = ["malfind.ninjections", "malfind.commitCharge", "malfind.uniqueInjections",
                  "psxview.not_in_pslist", "psxview.not_in_eprocess_pool", "psxview.not_in_ethread_pool", "psxview.not_in_pspcid_list",
                  "psxview.not_in_csrss_handles", "psxview.not_in_session", "psxview.not_in_deskthrd",
                  "ldrmodules.not_in_load", "ldrmodules.not_in_init", "ldrmodules.not_in_mem", "ldrmodules.not_in_load_avg",
                  "pslist.nproc", "dlllist.ndlls", "handles.nhandles", "handles.nmutant",
                  "svcscan.nservices", "svcscan.kernel_drivers", "callbacks.ncallbacks", "callbacks.nanonymous"]
GLOSS = {
    "malfind": "memory regions that are executable+writable and private (typical of injected / unpacked code)",
    "psxview": "cross-view process comparison: processes missing from some enumeration views while present in others (hidden / unlinked processes); '_false_avg' = ratio",
    "ldrmodules": "DLLs absent from the loader's load/init/mem lists (unlinked or manually mapped modules); '_avg' = ratio per process",
    "handles": "open kernel-object handle counts by type", "pslist": "process-list statistics", "dlllist": "loaded-DLL statistics",
    "svcscan": "Windows services and drivers", "callbacks": "kernel notification callbacks", "modules": "loaded kernel modules"}
SYSTEM = ("You are a senior Windows memory-forensics analyst. You receive aggregate indicators extracted with Volatility from ONE memory "
          "snapshot of a virtual machine, each compared with a reference distribution from clean snapshots. Decide whether the snapshot "
          "shows malware infection. Answer with exactly one word: MALICIOUS or BENIGN.")

def _f(v): return f"{v:.3g}"
def build_prompt(x, idx, ref, cols):
    lines, plugins = [], []
    for i in idx:
        v = x[i]; st = "ABOVE benign 95th pct" if v > ref["q95"][i] else ("BELOW benign 5th pct" if v < ref["q05"][i] else "within benign range")
        lines.append(f"- {cols[i]} = {_f(v)} (benign median {_f(ref['med'][i])}, 5th-95th pct {_f(ref['q05'][i])}-{_f(ref['q95'][i])}) -> {st}")
        p = cols[i].split(".")[0]; plugins += [p] if p not in plugins else []
    gloss = "\n".join(f"* {p}: {GLOSS[p]}" for p in plugins if p in GLOSS)
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Plugin glossary:\n{gloss}\n\nSnapshot indicators:\n" + "\n".join(lines) +
                                         "\n\nIs this snapshot MALICIOUS or BENIGN? Answer with one word."}]

def sigmoid(z): return 1 / (1 + np.exp(-np.asarray(z, float)))
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def ece_score(y, p, n_bins=10):
    b = np.digitize(p, np.linspace(0, 1, n_bins + 1)[1:-1]); e = 0.0
    for k in range(n_bins):
        m = b == k
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)

class MockBackend:
    """Plumbing test only: a keyword heuristic, NOT a language model. Never report its numbers."""
    name = "MOCK"
    def score(self, msgs_list):
        c = np.array([m[1]["content"] for m in msgs_list]); n_above = np.array([s.count("-> ABOVE") for s in c], float); n_all = np.array([s.count("-> ") for s in c], float)
        return sigmoid(1.1 * (n_above - 0.3 * n_all))
    def explain(self, msgs): return "[mock backend: no rationale generated]"

class HFBackend:
    def __init__(self, model_id, load_4bit=True):
        pk = [m for m in ("transformers", "accelerate", "bitsandbytes") if importlib.util.find_spec(m) is None]
        if pk:
            import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pk], check=False)
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        self.torch, self.name = torch, model_id
        self.tok = AutoTokenizer.from_pretrained(model_id); self.tok.padding_side = "left"
        if self.tok.pad_token is None: self.tok.pad_token = self.tok.eos_token
        kw = dict(device_map={"": 0})
        if load_4bit:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
        else:
            kw["torch_dtype"] = torch.float16      # T4 has no bf16
        self.model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
        self.id_mal = self.tok.encode("MALICIOUS", add_special_tokens=False)[0]
        self.id_ben = self.tok.encode("BENIGN", add_special_tokens=False)[0]
        assert self.id_mal != self.id_ben, "Answer words share their first token - change the answer vocabulary."
    def _text(self, msgs): return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    def score(self, msgs_list):
        torch = self.torch; texts = [self._text(m) for m in msgs_list]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
        for s in range(0, len(texts), LLM_BATCH):
            ids = order[s:s + LLM_BATCH]
            enc = self.tok([texts[i] for i in ids], return_tensors="pt", padding=True, truncation=True, max_length=2048).to("cuda:0")
            with torch.no_grad():
                try: lg = self.model(**enc, logits_to_keep=1).logits[:, -1, :]
                except TypeError:
                    h = self.model.model(**enc).last_hidden_state[:, -1]; lg = self.model.lm_head(h)
            out[ids] = sigmoid((lg[:, self.id_mal] - lg[:, self.id_ben]).float().cpu().numpy())
        return out
    def explain(self, msgs, max_new_tokens=110):
        torch = self.torch
        m = msgs[:-1] + [{"role": "user", "content": msgs[-1]["content"].replace("Answer with one word.", "Give the verdict, then justify it in two sentences citing the indicators.")}]
        enc = self.tok(self._text(m), return_tensors="pt").to("cuda:0")
        with torch.no_grad(): g = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        return self.tok.decode(g[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
LLM_RES = S_llm = None
if RUN_LLM and LLM_SAMPLE is not None and PAYLOAD:
    SFX = "_MOCK" if LLM_BACKEND == "mock" else ""
    if SFX: print("!!! MOCK LLM backend: plumbing test only, results are meaningless !!!")
    S = LLM_SAMPLE; ref = S["ref"]; idx = [COLS.index(n) for n in LLM_INDICATORS if n in COLS]
    print(f"{len(idx)}/{len(LLM_INDICATORS)} indicators present in the data")
    try: backend = MockBackend() if LLM_BACKEND == "mock" else HFBackend(LLM_MODEL_ID, LLM_4BIT)
    except Exception as e:
        raise RuntimeError(f"Could not load the LLM ({LLM_MODEL_ID}): {e}\nEnable Internet, or attach the model via Add Input -> Models, or set RUN_LLM = False / LLM_MODEL_ID to a smaller model.") from e
    pr_ = lambda Z: [build_prompt(x, idx, ref, COLS) for x in Z]
    print("Example evidence sheet (malware row):\n", pr_(X_[S["T"][next(iter(PAYLOAD))]][:1])[0][1]["content"][:900])
    fold_units = list(PAYLOAD); val_used = sorted({g for u in fold_units for g in PAYLOAD[u]["val_fams"]})
    t0 = time.time(); n_scored = 0
    def score_rows(Z):
        global n_scored; n_scored += len(Z); return backend.score(pr_(Z))
    sc_Bte, sc_Bva = score_rows(X_[S["B_te"]]), score_rows(X_[S["B_va"]])
    sc_T = {u: score_rows(X_[S["T"][u]]) for u in fold_units}; sc_C = {g: score_rows(X_[S["C"][g]]) for g in val_used}
    tmask = np.isin(S["T_fam"], fold_units); sc_EV = {}
    for c in LLM_CONDITIONS:
        full = np.full(len(S["T_fam"]), np.nan); full[tmask] = score_rows(S["EV"][c][tmask]); sc_EV[c] = full
    lat_B = (time.time() - t0) / n_scored * 1000
    print(f"LLM scoring done: {n_scored} snapshots, {lat_B:.0f} ms / snapshot")

    rows, pooled = [], dict(y=[], B_raw=[], B_platt=[], A=[]); LLM_ALL = MODELS + [M_B]
    for u in fold_units:
        pl = PAYLOAD[u]; msk = S["T_fam"] == u
        cal_y = np.r_[np.zeros(len(sc_Bva), int), np.ones(sum(len(sc_C[g]) for g in pl["val_fams"]), int)]; cal_s = np.concatenate([sc_Bva] + [sc_C[g] for g in pl["val_fams"]])
        tauB = calibrate_thresholds(cal_y, cal_s); platt = LogisticRegression(C=1e6).fit(logit(cal_s).reshape(-1, 1), cal_y)
        bank = {M_B: dict(tau=tauB, ben=sc_Bte, clean=sc_T[u], ev={c: sc_EV[c][msk] for c in LLM_CONDITIONS})}
        for nm in MODELS: bank[nm] = dict(tau=pl["tau"][nm], ben=pl["benign"][nm], clean=pl["clean"][nm], ev={c: pl["ev"][(nm, c)] for c in LLM_CONDITIONS})
        for nm, d in bank.items():
            for kind in TAU_KINDS:
                t_ = d["tau"][kind]; yy_ = np.r_[np.zeros(len(d["ben"]), int), np.ones(len(d["clean"]), int)]
                rows.append(dict(unit=u, model=nm, tau_kind=kind, condition="clean", alpha=0.0, tau=t_, **binary_metrics(yy_, np.r_[d["ben"], d["clean"]], t_)))
                for (tech, al), se in d["ev"].items():
                    yy2 = np.r_[np.zeros(len(d["ben"]), int), np.ones(len(se), int)]
                    rows.append(dict(unit=u, model=nm, tau_kind=kind, condition=tech, alpha=al, tau=t_, **binary_metrics(yy2, np.r_[d["ben"], se], t_)))
        yy_ = np.r_[np.zeros(len(sc_Bte), int), np.ones(len(sc_T[u]), int)]; pooled["y"].append(yy_)
        pooled["B_raw"].append(np.r_[sc_Bte, sc_T[u]]); pooled["B_platt"].append(platt.predict_proba(logit(np.r_[sc_Bte, sc_T[u]]).reshape(-1, 1))[:, 1])
        pooled["A"].append(np.r_[pl["benign"][M_A], pl["clean"][M_A]])
    LLM_RES = pd.DataFrame(rows); LLM_RES["alpha_key"] = LLM_RES.alpha
    KEYL = ["unit", "model", "tau_kind"]; rc = LLM_RES[LLM_RES.condition == "clean"][KEYL + ["recall"]].rename(columns={"recall": "recall_clean"})
    LLM_RES = LLM_RES.merge(rc, on=KEYL); LLM_RES["eidd"] = np.where((LLM_RES.condition != "clean") & (LLM_RES.recall_clean > 0), (LLM_RES.recall_clean - LLM_RES.recall) / LLM_RES.recall_clean * 100, np.nan)
    S_llm = summarize(LLM_RES, ["model", "tau_kind", "condition", "alpha"], "recall"); S_llm_e = summarize(LLM_RES[LLM_RES.condition != "clean"], ["model", "tau_kind", "condition", "alpha"], "eidd")
    S_llm_c = summarize(LLM_RES[LLM_RES.condition == "clean"].melt(id_vars=KEYL, value_vars=["auc", "recall", "fpr", "f1", "mcc"], var_name="metric", value_name="v"), ["model", "tau_kind", "metric"], "v")
    y_p, pB, pBp, pA = (np.concatenate(pooled[k]) for k in ("y", "B_raw", "B_platt", "A"))
    cal = pd.DataFrame([dict(model=M_B + " raw", ece=ece_score(y_p, pB), brier=brier_score_loss(y_p, pB)),
                        dict(model=M_B + " + Platt (leave-family-out)", ece=ece_score(y_p, pBp), brier=brier_score_loss(y_p, pBp)),
                        dict(model=M_A + " raw", ece=ece_score(y_p, pA), brier=brier_score_loss(y_p, pA))])
    lat = pd.DataFrame([dict(model=M_B, ms_per_snapshot=lat_B), dict(model=M_A, ms_per_snapshot=float(FOLDS[FOLDS.protocol == "P3_LOFO"].ms_per_snapshot_A.mean()))])
    LLM_RES.to_csv(OUT / f"llm_results_per_fold{SFX}.csv", index=False); S_llm.to_csv(OUT / f"llm_summary_recall{SFX}.csv", index=False); S_llm_e.to_csv(OUT / f"llm_summary_eidd{SFX}.csv", index=False)
    S_llm_c.to_csv(OUT / f"llm_summary_clean{SFX}.csv", index=False); cal.to_csv(OUT / f"llm_calibration{SFX}.csv", index=False); lat.to_csv(OUT / f"llm_latency{SFX}.csv", index=False)
    pd.DataFrame(dict(y=y_p, p_B=pB, p_B_platt=pBp, p_A=pA)).to_csv(OUT / f"llm_scores_pooled{SFX}.csv", index=False)

    # qualitative rationales: first caught and first two missed evaded rows (first fold with such rows)
    ex = []; c0 = LLM_CONDITIONS[0]
    for u in fold_units:
        se = sc_EV[c0][S["T_fam"] == u]; tB = calibrate_thresholds(np.r_[np.zeros(len(sc_Bva), int), np.ones(sum(len(sc_C[g]) for g in PAYLOAD[u]["val_fams"]), int)],
                                                                   np.concatenate([sc_Bva] + [sc_C[g] for g in PAYLOAD[u]["val_fams"]]))["Youden"]
        Xe_u = S["EV"][c0][S["T_fam"] == u]; caught, missed = np.where(se >= tB)[0], np.where(se < tB)[0]
        for tag, ids in (("evaded-caught", caught[:1] if not any(e["case"] == "evaded-caught" for e in ex) else []), ("evaded-missed", missed[:2] if sum(e["case"] == "evaded-missed" for e in ex) < 2 else [])):
            for i in ids:
                m_ = build_prompt(Xe_u[i], idx, ref, COLS); ex.append(dict(case=tag, family=u, condition=f"{c0[0]} a={c0[1]}", p_malicious=float(se[i]), evidence=m_[1]["content"], rationale=backend.explain(m_)))
        if len(ex) >= 3: break
    (OUT / f"llm_examples{SFX}.json").write_text(json.dumps(ex, indent=2))

    conds = [("clean", 0.0)] + list(LLM_CONDITIONS)
    print(f"\n== Recall at frozen tau, mean over {len(fold_units)} folds (Youden) ==")
    t = S_llm[S_llm.tau_kind == "Youden"].assign(c=lambda d: d.condition + "@" + d.alpha.astype(str)).pivot(index="model", columns="c", values="mean")
    print(t.reindex(LLM_ALL).round(3).to_string()); print("\nClean metrics (Youden):"); print(S_llm_c[S_llm_c.tau_kind == "Youden"].pivot(index="model", columns="metric", values="mean").reindex(LLM_ALL).round(3).to_string())
    print(cal.round(4).to_string(index=False)); print(lat.round(2).to_string(index=False))

    make_fig11(S_llm, y_p, pB, pBp, pA, SFX)
else:
    print("RUN_LLM = False (or no LOFO payload): skipping Model B.")

## 10. Export: configuration, headline numbers, reproducibility record
Everything needed for the paper's *Experimental setup* and *Reproducibility* sections is written to `outputs/`. **`run_record.json` also stores the feasibility report** (which structural constraints were verified on the training rows and adopted). If few constraints were adopted, the real column semantics differ from the assumed ones and the feasibility projection is correspondingly weak: state this in the paper.

In [ ]:
import shutil
def _num(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    return str(o)
head = {}
for name in MODELS:
    def r(df, **kw): return df[(df.protocol == PRIM) & (df.tau_kind == "Youden") & (df.model == name) & np.all([df[k] == v for k, v in kw.items()], axis=0)]
    head[name] = dict(
        auc_clean=r(S_clean, metric="auc")["mean"].iloc[0], recall_clean=r(S_clean, metric="recall")["mean"].iloc[0], fpr_clean=r(S_clean, metric="fpr")["mean"].iloc[0],
        resilience_at_primary_alpha=r(S_res, alpha=PRIMARY_ALPHA)["mean"].iloc[0], resilience_at_alpha_1=r(S_res, alpha=1.0)["mean"].iloc[0],
        worst_case_recall_primary_alpha=r(S_wc, alpha=PRIMARY_ALPHA)["mean"].iloc[0], recall_combined_primary_alpha=r(S_rec, condition="Combined", alpha=PRIMARY_ALPHA)["mean"].iloc[0])
feas = next(iter(ART.values()))["feas"]
record = dict(smoke_test=SMOKE_TEST, fast_preset=FAST, n_folds=int(N_U), folds=FOLD_FAMS,
              config=dict(SEED0=SEED0, PROTOCOLS=PROTOCOLS, N_SEEDS_P1=N_SEEDS_P1, LOFO_N_VAL_FAMILIES=LOFO_N_VAL_FAMILIES, ALPHAS=ALPHAS, PRIMARY_ALPHA=PRIMARY_ALPHA, FPR_TARGET=FPR_TARGET,
                          AUG_FRAC=AUG_FRAC, JITTER=JITTER, FEASIBLE=FEASIBLE, FEAS_HOLD=FEAS_HOLD, XGB_N_EST=XGB_N_EST, RF_N_EST=RF_N_EST, ADAPT=ADAPT, ADAPT_N=ADAPT_N, ADAPT_KMAX=ADAPT_KMAX,
                          SENS_JITTER=SENS_JITTER, LLM_BACKEND=LLM_BACKEND if RUN_LLM else None, LLM_MODEL_ID=LLM_MODEL_ID, LLM_INDICATORS=LLM_INDICATORS,
                          LLM_N_PER_FAMILY=LLM_N_PER_FAMILY, LLM_N_BENIGN=LLM_N_BENIGN, techniques={t: [COLS[i] for i in TECH_IDX[t]] for t in TECHS}),
              data=D["info"], feasibility=feas, headline_primary_protocol=head, fold_seconds=FOLDS.seconds.sum(),
              environment=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, xgboost=xgb.__version__, matplotlib=mpl.__version__))
(OUT / "run_record.json").write_text(json.dumps(record, indent=2, default=_num))
print(json.dumps(head, indent=2, default=_num))
if feas is not None: print("\nFeasibility constraints adopted:", {k: (len(v) if isinstance(v, list) else v) for k, v in feas["adopted"].items()})
if SMOKE_TEST: print("\n!!! SMOKE_TEST run: synthetic data - do NOT use any of these numbers in a paper. !!!")
import pickle
with open(OUT / "art_scores.pkl", "wb") as fh: pickle.dump(dict(ART=ART, PAYLOAD=PAYLOAD if "PAYLOAD" in globals() else None), fh)   # lets figures 1, 5, 7 be re-drawn without re-running the folds
shutil.make_archive(str(OUT.parent / "memforensics_results"), "zip", OUT)
print("Archive:", OUT.parent / "memforensics_results.zip")